# 03 — 지역 간 전이 실험 (본체)**주장 한 문장**> 위성 기반 산림 분류를 다른 지역에 옮기면 분류 정확도는 완만하게 떨어지는데> **탄소 추정 오차는 훨씬 크게, 한 방향으로 치우친다.**> 그 방향은 **지역 간 분광 오프셋**으로 사전에 예측된다.## ⚠️ 실행 전 필수```pythonC = {1: 79.25, 2: 101.02, 3: 92.01}   # 강원 NFI 2013 / GIR 공표 (tC/ha)````[R]` 복구 블록의 C를 확정값으로 이미 바꿔뒀습니다. **`[4]`/`[5]`/`[6]`의 C는 구 임시값(55/65/60) 그대로**인데,이 셀들의 출력이 보고서에 "구 계수 기준"으로 인용돼 있어 일부러 남겼습니다.`[6]`이 **예측을 저장**하므로 이후 분석은 **재학습 없이 계수 한 줄만 바꿔** 갱신됩니다.## 읽는 순서| 단계 | 셀 | 내용 ||---|---|---|| 준비 | `[1]` `[2]` `[3]` | 블록 단위 샘플 추출 → 캐시 → 추출 코드 검증 || **주장 ①②** | `[4]` `[5]` `[6]` | 전이 9조합 학습·예측 저장 || **주장 ②** | `[7]` | 혼동행렬 → 계수 가중 오차 분해 (**9칸 표의 출처**) || 실사용 | `[8]` | 타일(사업지) 단위 오차 분포 || 메커니즘 | `[10]` `[11]` `[18]` | 낙엽송 기여도 → **노출도 통제 실험** || 가설 기각 | `[12]` `[13]` `[16]` | 수종 다양성 / 분광 커버리지 / 정규화 — **전부 실패** || **주장 ③** | `[14]` `[15]` `[17]` | **지역 오프셋 발견 → 오차 방향 예측 (상관 0.933)** || 확정 | `[19]` `[20]` | 닫힌 형태 검산, 확정 계수 적용 || 미실행 | `[21]` | 타일 오차 분포 그림 |## 남은 작업 (보고서 8절)1. **공간 분리 재설계** (45분) — 지역마다 동서로 나눠 서쪽 학습 → 자기 동쪽(대각) / 남의 동쪽(비대각).   현재 대각은 랜덤 타일이라 인접 타일 덕을 봅니다. **배율 주장이 여기 걸려 있습니다.**2. **위약 대조** (2시간) — 낙엽송 고정, 상록 침엽끼리(소나무↔잣나무) 같은 폭 조작. 안 움직여야 특이성 입증.3. 시드 5개 재실행 (35분) · 로지스틱 회귀 대조 (30분) · `[21]` 실행 (5분)

# 03 — 지역 간 전이 실험> **다른 지역에서 학습한 모델을 가져다 쓰면 어떻게 되는가?**`01`에서 대전 한 지역은 잘 됐고, 같은 지역 안에서도 **공간을 분리하니 오차가 10배**로늘었습니다(`01-[U]`). 그래서 지역을 아예 바꿔서 확인합니다.### 실행 전 필수`[R]` 을 먼저 실행하세요. 공통 함수와 **확정 탄소계수**를 세팅합니다.```pythonC = {1: 79.25, 2: 101.02, 3: 92.01}   # tC/ha, 강원 NFI 2013 / GIR````[4]` `[5]` `[6]` `[15]` `[16]` 은 **구 임시계수(55/65/60)** 로 실행된 출력을 담고 있습니다.보고서가 그 값을 인용하고 있어 일부러 남겼고, 전역 계수는 셀 끝에서 복구됩니다.### 읽는 순서| 단계 | 셀 | 내용 ||---|---|---|| 준비 | `[1]`~`[3]` | 블록 단위 샘플 추출 → 캐시 → 추출 코드 검증 || **전이** | `[4]`~`[7]` | 9조합 학습·예측 저장 → **오차 분해** || 실사용 | `[8]` `[9]` | 타일 단위 분포 · 계수 감도 || 원인 | `[10]` `[11]` `[18]` | 낙엽송 기여도 → **노출도 통제 실험** || 기각 | `[12]` `[13]` | 수종 다양성 · 분광 커버리지 || **핵심** | `[14]` `[15]` `[17]` | **지역 오프셋 → 오차 방향 예측** || 처방 | `[16]` | 지역별 정규화 || 확정 | `[19]` `[20]` | 닫힌 형태 검산 · 확정 계수 적용 || 미실행 | `[21]` | 타일 오차 분포 그림 |---## 준비

In [ ]:
# ============================================================
# [1] 셋업 및 지역별 샘플 추출
#   전체 이미지를 메모리에 올리지 않고 블록 단위로 읽으며
#   학습/평가에 쓸 픽셀만 뽑아 npz로 저장한다.
#   → 지역 크기와 무관하게 동작하고, 이후 실험이 즉시 로드된다
# ============================================================
!pip install -q rasterio

import numpy as np, rasterio, os
from rasterio.windows import Window
from scipy.ndimage import uniform_filter
from google.colab import drive

drive.mount('/content/drive')
D   = '/content/drive/MyDrive/forest/data'
OUT = '/content/drive/MyDrive/forest/outputs'
os.makedirs(OUT, exist_ok=True)
names_cls = ['침엽수림', '활엽수림', '혼효림']

def make_feats(X):
    """(20,h,w) 반사율 → (38,h,w) 피처. 01_explore 와 동일한 구성/순서"""
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    s_ndvi, w_ndvi = nd(X[6], X[2]),  nd(X[16], X[12])
    s_ndmi, w_ndmi = nd(X[6], X[8]),  nd(X[16], X[18])
    d_ndvi, d_ndmi = s_ndvi - w_ndvi, s_ndmi - w_ndmi
    f = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi,
                                     w_ndmi, d_ndvi, d_ndmi])])
    extra = []
    for base in (s_ndvi, w_ndvi, d_ndvi):
        for w in (5, 15):
            a  = np.nan_to_num(base)
            mu = uniform_filter(a, size=w)
            sd = np.sqrt(np.maximum(uniform_filter(a**2, size=w) - mu**2, 0))
            extra += [mu, sd]
    return np.concatenate([f, np.stack(extra)]).astype('float32')


def extract_samples(name, n_max=600_000, block=1024, pad=16, seed=0, TILE=500):
    """지역에서 유효 픽셀(산림 3클래스, 갱신년도 2024+)을 최대 n_max개 샘플링"""
    cache = f'{OUT}/{name}_samples.npz'
    if os.path.exists(cache):
        z = np.load(cache)
        print(f'{name}: 캐시 로드 {z["X"].shape}')
        return {k: z[k] for k in z.files}

    with rasterio.open(f'{D}/{name}_label.tif')   as s: label = s.read(1)
    with rasterio.open(f'{D}/{name}_year.tif')    as s: year  = s.read(1)
    with rasterio.open(f'{D}/{name}_species.tif') as s: sp    = s.read(1)

    H, W = label.shape
    use  = (label >= 1) & (label <= 3) & (year >= 2024)
    frac = min(1.0, n_max / max(use.sum(), 1))
    rng_b = np.random.default_rng(seed)
    Xs, ys, ts, ss = [], [], [], []

    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as src:
        scale = 10000.0 if src.dtypes[0] == 'int16' else 1.0
        for r0 in range(0, H, block):
            for c0 in range(0, W, block):
                r1, c1 = min(H, r0+block), min(W, c0+block)
                m = use[r0:r1, c0:c1]
                if not m.any(): continue
                sel = m & (rng_b.random(m.shape) < frac)
                if not sel.any(): continue

                pr0, pc0 = max(0, r0-pad), max(0, c0-pad)
                pr1, pc1 = min(H, r1+pad), min(W, c1+pad)
                X = src.read(window=Window(pc0, pr0, pc1-pc0, pr1-pr0))
                X = X.astype('float32') / scale
                f = make_feats(X)
                f = f[:, r0-pr0:r0-pr0+(r1-r0), c0-pc0:c0-pc0+(c1-c0)]

                rr, cc = np.where(sel)
                Xs.append(f[:, sel].T)
                ys.append(label[r0:r1, c0:c1][sel])
                ss.append(sp[r0:r1, c0:c1][sel])
                ts.append(((rr+r0)//TILE)*10000 + (cc+c0)//TILE)
                del X, f

    d = dict(X=np.concatenate(Xs), y=np.concatenate(ys),
             tile=np.concatenate(ts), sp=np.concatenate(ss))
    np.savez_compressed(cache, **d)
    print(f'{name}: {d["X"].shape[0]:,}개 추출 → 저장')
    return d

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# [2] 지역별 샘플 추출 (첫 실행만 오래 걸림, 이후 캐시)
#   홍천 1.6GB 를 블록 단위로 읽으므로 5~10분 소요 가능
# ============================================================
S = {n: extract_samples(n) for n in ['daejeon', 'hongcheon', 'suncheon']}
for n, d in S.items():
    u, c = np.unique(d['y'], return_counts=True)
    print(f'{n:10s} {d["X"].shape[0]:>8,}개 | '
          f'침엽 {c[0]:>7,} 활엽 {c[1]:>7,} 혼효 {c[2]:>7,}')

daejeon: 캐시 로드 (599719, 38)
hongcheon: 캐시 로드 (599527, 38)
suncheon: 캐시 로드 (600846, 38)
daejeon     599,719개 | 침엽 239,240 활엽 238,972 혼효 121,507
hongcheon   599,527개 | 침엽 225,716 활엽 294,060 혼효  79,751
suncheon    600,846개 | 침엽 311,835 활엽 227,890 혼효  61,121


### `[3]` 추출 코드 검증대전 지역 내 타일 분할에서 macro F1 0.62~0.63이 나오면 샘플 추출이 정상입니다.

In [ ]:
# ============================================================
# [3] 추출 코드 검증
#   대전 지역 내 타일 분할 → macro F1 0.62~0.63 이면 정상
#   (01_explore 의 0.628 과 샘플링 방식만 다르므로 완전 일치는 아님)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

d = S['daejeon']
rng = np.random.default_rng(42)
uniq = np.unique(d['tile']); rng.shuffle(uniq)
tr_t = set(uniq[:int(len(uniq)*0.7)].tolist())
m_tr = np.isin(d['tile'], list(tr_t))
print(f'학습 {m_tr.sum():,} / 평가 {(~m_tr).sum():,}')

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                            n_jobs=-1, class_weight='balanced', random_state=42)
rf.fit(d['X'][m_tr], d['y'][m_tr])
p = rf.predict(d['X'][~m_tr])

print(classification_report(d['y'][~m_tr], p, target_names=names_cls, digits=3))
print('macro F1:', round(f1_score(d['y'][~m_tr], p, average='macro'), 3),
      ' (01_explore 기준 0.628)')

학습 413,274 / 평가 186,445
              precision    recall  f1-score   support

        침엽수림      0.734     0.835     0.781     75797
        활엽수림      0.776     0.719     0.747     75703
         혼효림      0.361     0.310     0.334     34945

    accuracy                          0.690    186445
   macro avg      0.624     0.622     0.620    186445
weighted avg      0.681     0.690     0.683    186445

macro F1: 0.62  (01_explore 기준 0.628)


## 주장 ①② — 전이 9조합### `[4]` 지역 간 전이 실험학습 지역과 평가 지역을 바꿔가며 측정. 학습량 20만으로 통일.**F1은 0.626 → 0.551로 12%만 떨어집니다.** 겉보기로는 쓸 만해 보입니다.> ⚠️ `rng_g`는 모듈 레벨 난수라 실행 순서에 의존합니다.

---## 전이 실험학습 지역과 평가 지역을 바꿔가며 9조합을 측정합니다.**학습량은 20만 픽셀로 통일**해 조건을 맞춥니다.

In [ ]:
# ============================================================
# [4] 지역 간 전이 실험
#   학습 지역과 평가 지역을 바꿔가며 일반화 성능을 측정한다.
#   - 지역 내(within): 같은 지역 타일 분할. 상한선 역할
#   - 지역 간(cross) : 다른 지역에서 평가. 실제 활용 상황
#   학습량을 20만으로 통일해 조건을 맞춘다.
# ============================================================
import itertools, time
from sklearn.metrics import confusion_matrix

REGIONS = ['daejeon', 'hongcheon', 'suncheon']
N_TRAIN, N_TEST = 200_000, 150_000
rng_g = np.random.default_rng(0)

def sub(d, mask, n):
    idx = np.where(mask)[0]
    if len(idx) > n:
        idx = rng_g.choice(idx, n, replace=False)
    return d['X'][idx], d['y'][idx]

def fit_eval(Xtr, ytr, Xte, yte, tag):
    t0 = time.time()
    m = RandomForestClassifier(n_estimators=150, min_samples_leaf=5,
                               n_jobs=-1, class_weight='balanced',
                               random_state=42)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    f1c = f1_score(yte, p, average=None, labels=[1,2,3])
    res = dict(tag=tag, macro=f1_score(yte, p, average='macro'),
               conif=f1c[0], broad=f1c[1], mixed=f1c[2],
               n_tr=len(ytr), n_te=len(yte), sec=time.time()-t0)
    print(f'{tag:28s} macro {res["macro"]:.3f} | '
          f'침엽 {res["conif"]:.3f} 활엽 {res["broad"]:.3f} '
          f'혼효 {res["mixed"]:.3f}  ({res["sec"]:.0f}s)')
    return res, m, p

results = []

# --- 지역 내 (상한선) ---
print('=== 지역 내 (타일 분할) ===')
for r in REGIONS:
    d = S[r]
    u = np.unique(d['tile']); np.random.default_rng(42).shuffle(u)
    m_tr = np.isin(d['tile'], list(u[:int(len(u)*0.7)]))
    Xtr, ytr = sub(d,  m_tr, N_TRAIN)
    Xte, yte = sub(d, ~m_tr, N_TEST)
    results.append(fit_eval(Xtr, ytr, Xte, yte, f'{r} → {r}')[0])

# --- 지역 간 ---
print('\n=== 지역 간 (1 → 1) ===')
for a, b in itertools.permutations(REGIONS, 2):
    Xtr, ytr = sub(S[a], np.ones(len(S[a]['y']), bool), N_TRAIN)
    Xte, yte = sub(S[b], np.ones(len(S[b]['y']), bool), N_TEST)
    results.append(fit_eval(Xtr, ytr, Xte, yte, f'{a[:3]} → {b[:3]}')[0])

# --- 2지역 학습 → 1지역 평가 ---
print('\n=== 2지역 학습 → 1지역 평가 ===')
for held in REGIONS:
    tr_r = [r for r in REGIONS if r != held]
    Xs, ys = [], []
    for r in tr_r:                      # 지역당 절반씩 → 한쪽에 치우치지 않게
        x, y_ = sub(S[r], np.ones(len(S[r]['y']), bool), N_TRAIN//2)
        Xs.append(x); ys.append(y_)
    Xte, yte = sub(S[held], np.ones(len(S[held]['y']), bool), N_TEST)
    results.append(fit_eval(np.concatenate(Xs), np.concatenate(ys),
                            Xte, yte, f'2지역 → {held[:3]}')[0])

=== 지역 내 (타일 분할) ===
daejeon → daejeon            macro 0.618 | 침엽 0.781 활엽 0.744 혼효 0.329  (303s)
hongcheon → hongcheon        macro 0.620 | 침엽 0.726 활엽 0.798 혼효 0.335  (296s)
suncheon → suncheon          macro 0.610 | 침엽 0.818 활엽 0.792 혼효 0.220  (272s)

=== 지역 간 (1 → 1) ===
dae → hon                    macro 0.556 | 침엽 0.674 활엽 0.774 혼효 0.219  (307s)
dae → sun                    macro 0.564 | 침엽 0.778 활엽 0.686 혼효 0.229  (296s)
hon → dae                    macro 0.594 | 침엽 0.735 활엽 0.695 혼효 0.353  (293s)
hon → sun                    macro 0.555 | 침엽 0.797 활엽 0.664 혼효 0.205  (294s)
sun → dae                    macro 0.554 | 침엽 0.728 활엽 0.726 혼효 0.208  (273s)
sun → hon                    macro 0.449 | 침엽 0.530 활엽 0.731 혼효 0.087  (272s)

=== 2지역 학습 → 1지역 평가 ===
2지역 → dae                    macro 0.598 | 침엽 0.740 활엽 0.729 혼효 0.324  (300s)
2지역 → hon                    macro 0.542 | 침엽 0.648 활엽 0.765 혼효 0.212  (290s)
2지역 → sun                    macro 0.577 | 침엽 0.808 활엽 0.690 혼효 0.234  (29

In [ ]:
# ============================================================
# [5] 지역 간 전이 시 탄소 오차
#   지역 내에서는 ±1% 이내였다. 전이 상황에서도 유지되는가?
#   분류 성능은 -11.5% 떨어졌지만 탄소 추정은 다를 수 있다.
# ============================================================
C = {1: 55.0, 2: 65.0, 3: 60.0}      # 구 임시값 유지 — 이 출력이 보고서에 '구 계수 기준'으로 인용됨
PIX_HA = 0.01

def carbon_err(y_true, y_pred):
    t = sum((y_true == v).sum() * PIX_HA * C[v] for v in (1,2,3))
    p = sum((y_pred == v).sum() * PIX_HA * C[v] for v in (1,2,3))
    return (p - t) * 100 / t

print(f'{"조합":18s}{"macro":>8s}{"탄소오차":>10s}')
for a, b in itertools.permutations(REGIONS, 2):
    Xtr, ytr = sub(S[a], np.ones(len(S[a]['y']), bool), N_TRAIN)
    Xte, yte = sub(S[b], np.ones(len(S[b]['y']), bool), N_TEST)
    m = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                               max_features='sqrt', n_jobs=-1,
                               class_weight='balanced', random_state=42)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    print(f'{a[:3]} → {b[:3]:12s}'
          f'{f1_score(yte, p, average="macro"):8.3f}'
          f'{carbon_err(yte, p):9.2f}%')

조합                   macro      탄소오차
dae → hon            0.565     2.01%
dae → sun            0.553    -1.29%
hon → dae            0.582    -1.23%
hon → sun            0.544    -2.50%
sun → dae            0.587     2.05%
sun → hon            0.481     4.20%


### `[6]` 예측 저장 ★조합마다 독립 시드. **이후 분석은 재학습 없이 이 npz만 씁니다.**> 📌 `random_state=42`와 타일 분할 시드 42가 **둘 다 고정**입니다. `[5]`와의 차이는 **어느 픽셀을 뽑았는가 하나뿐**이라,> 여기서 나온 노이즈 0.11%p는 샘플링 변동만 잰 하한값입니다. (보고서 7-2)

### `[6]` 예측 저장 — 이후 분석은 재학습 없이 이 결과만 사용> `random_state` 와 타일 분할 시드가 **둘 다 고정**입니다.> `[5]` 와의 차이는 **어느 픽셀을 뽑았는가** 하나뿐이라,> 여기서 얻은 노이즈 기준선은 **하한값**입니다.

In [ ]:
# ============================================================
# [6] 예측 저장 — 이후 분석은 재학습 없이 이 결과만 사용
#   - 조합마다 독립 시드 (rng_g 전역 상태 의존 제거 → 재현 가능)
#   - 지역 내 3조합 포함 (전이 비교의 기준선)
#   - 경량/정식 두 설정 (전이에서 경량이 나은지 확인)
#   - 중단되면 이어서 실행 (이미 저장된 조합은 건너뜀)
# ============================================================
import json, time, os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

PRED = f'{OUT}/preds'; os.makedirs(PRED, exist_ok=True)

C = {1: 55.0, 2: 65.0, 3: 60.0}      # 구 임시값 유지 — 이 출력이 보고서에 '구 계수 기준'으로 인용됨
def carbon_err(y_true, y_pred):
    t = sum((y_true == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((y_pred == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100 / t

RF_BASE = dict(n_jobs=-1, class_weight='balanced', random_state=42)
CFGS = {'light': dict(n_estimators=80, min_samples_leaf=20, max_features='sqrt'),
        'full' : dict(n_estimators=150, min_samples_leaf=5)}

def sub2(d, mask, n, seed):
    idx = np.where(mask)[0]
    if len(idx) > n:
        idx = np.random.default_rng(seed).choice(idx, n, replace=False)
    return {k: d[k][idx] for k in ('X', 'y', 'tile', 'sp')}

def within_mask(d):                      # 셀 [4]와 동일한 타일 분할
    u = np.unique(d['tile']); np.random.default_rng(42).shuffle(u)
    return np.isin(d['tile'], list(u[:int(len(u)*0.7)]))

def run_save(a, b, cfg_name):
    tag = f'{a[:3]}_{b[:3]}_{cfg_name}'
    if os.path.exists(f'{PRED}/{tag}.npz'):
        print(f'{tag:22s} 건너뜀 (이미 있음)'); return
    if a == b:
        m = within_mask(S[a])
        tr, te = sub2(S[a], m, N_TRAIN, 1), sub2(S[a], ~m, N_TEST, 2)
    else:
        tr = sub2(S[a], np.ones(len(S[a]['y']), bool), N_TRAIN, 1)
        te = sub2(S[b], np.ones(len(S[b]['y']), bool), N_TEST,  2)
    t0 = time.time()
    rf = RandomForestClassifier(**CFGS[cfg_name], **RF_BASE).fit(tr['X'], tr['y'])
    p  = rf.predict(te['X'])
    np.savez_compressed(f'{PRED}/{tag}.npz', y_true=te['y'],
                        y_pred=p.astype('int16'), tile=te['tile'], sp=te['sp'],
                        cfg=json.dumps({**CFGS[cfg_name], **RF_BASE}))
    print(f'{tag:22s} macro {f1_score(te["y"], p, average="macro"):.3f}'
          f'  탄소 {carbon_err(te["y"], p):+6.2f}%  ({time.time()-t0:.0f}s)')

for nm in ('light', 'full'):
    print(f'\n=== {nm} ===')
    for a in REGIONS:
        for b in REGIONS:
            run_save(a, b, nm)

print(f'\n저장 완료: {len(os.listdir(PRED))}/18')


=== light ===
dae_dae_light          macro 0.624  탄소  -0.81%  (169s)
dae_hon_light          macro 0.569  탄소  +1.92%  (143s)
dae_sun_light          macro 0.550  탄소  -1.34%  (144s)
hon_dae_light          macro 0.581  탄소  -1.26%  (145s)
hon_hon_light          macro 0.625  탄소  -0.02%  (147s)
hon_sun_light          macro 0.539  탄소  -2.55%  (139s)
sun_dae_light          macro 0.583  탄소  +2.16%  (132s)
sun_hon_light          macro 0.484  탄소  +4.21%  (139s)
sun_sun_light          macro 0.630  탄소  +0.51%  (142s)

=== full ===
dae_dae_full           macro 0.620  탄소  -0.74%  (331s)
dae_hon_full           macro 0.552  탄소  +1.80%  (347s)
dae_sun_full           macro 0.560  탄소  -1.61%  (340s)
hon_dae_full           macro 0.595  탄소  -0.93%  (327s)
hon_hon_full           macro 0.621  탄소  +0.20%  (331s)
hon_sun_full           macro 0.558  탄소  -2.49%  (325s)
sun_dae_full           macro 0.546  탄소  +2.13%  (300s)
sun_hon_full           macro 0.448  탄소  +4.17%  (300s)
sun_sun_full           macro 0.611  

### `[R]` 런타임 복구 + 공통 함수**계수를 확정값(79.25 / 101.02 / 92.01)으로 바꿔뒀습니다.** 이후 셀 전에 한 번만 실행하세요.

In [ ]:
# ============================================================
# [R] 런타임 복구 + 공통 함수 — 이후 셀들 전에 한 번만 실행
# ============================================================
import numpy as np, os
from sklearn.metrics import confusion_matrix, f1_score
from google.colab import drive; drive.mount('/content/drive')

OUT  = '/content/drive/MyDrive/forest/outputs'
PRED = f'{OUT}/preds'
REGIONS = ['daejeon', 'hongcheon', 'suncheon']
C_CONF = {1: 79.25, 2: 101.02, 3: 92.01}   # 확정: 강원 NFI 2013 / GIR
C = dict(C_CONF)                           # g=21.77, δ=+1.87
CFG  = 'light'
NAME = {1: '침엽', 2: '활엽', 3: '혼효'}
NOISE = 0.11          # 탄소오차 재현 노이즈 (%p). [5] vs [6] 실측
NOISE_F1 = 0.005      # macro F1 재현 노이즈

def carbon_err(y_true, y_pred):
    t = sum((y_true == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((y_pred == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100 / t

def path(a, b, cfg=CFG):
    return f'{PRED}/{a[:3]}_{b[:3]}_{cfg}.npz'

def have(a, b, cfg=CFG):
    return os.path.exists(path(a, b, cfg))

def load_pred(a, b, cfg=CFG):
    z = np.load(path(a, b, cfg))
    return {k: z[k] for k in z.files}

def pairs(cfg=CFG):
    return [(a, b) for a in REGIONS for b in REGIONS if have(a, b, cfg)]

for cfg in ('light', 'full'):
    n = sum(1 for a in REGIONS for b in REGIONS if have(a, b, cfg))
    print(f'{cfg:6s} {n}/9')

d = load_pred(*pairs()[0])
print(f'\n키: {list(d.keys())}')
print(f'평가셋 크기: {d["y_true"].size:,}   타일 수: {np.unique(d["tile"]).size}')
print(f'검산: carbon_err = {carbon_err(d["y_true"], d["y_pred"]):+.2f}%  (기대 -0.81)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
light  9/9
full   9/9

키: ['y_true', 'y_pred', 'tile', 'sp', 'cfg']
평가셋 크기: 150,000   타일 수: 12
검산: carbon_err = -0.81%  (기대 -0.81)


## 주장 ② — 오차의 방향성### `[7]` 오차 분해 ★★ — 보고서 4-② 9칸 표의 출처**총오차(오분류 절대량)는 4.55 → 7.02%로 1.25배밖에 안 느는데, 순오차(실제 탄소 오차)는 0.12 → 5.92%로 뜁니다.**지역 내에서는 과대·과소 오분류가 상쇄되는데, 전이하면 오차가 한 방향으로 쏠려 상쇄가 깨집니다.> 가법모델(행효과+열효과)은 **기각**됐습니다. 잔차 대각 3개가 전부 음수, 비대각 6개가 전부 양수입니다.

### `[7]` 오차 분해 — 어느 혼동이 탄소 오차를 만드는가**총오차(오분류 절대량)는 거의 안 느는데 순오차(탄소)만 크게 뜁니다.**지역 내에서는 과대·과소 오분류가 상쇄되는데, 전이하면 한 방향으로 쏠려 상쇄가 깨집니다.

In [ ]:
# ============================================================
# [7] 오차 분해 — 어느 혼동이 탄소 오차를 만드는가
#   지역 내 오차가 작았던 건 인접 클래스 혼동이 상쇄됐기 때문.
#   전이에서 상쇄가 깨졌다면 침엽↔활엽(계수차 최대)이 한 방향으로 몰린 것.
#
#   전제: 이 셀 위에서 C 와 NOISE 를 확정값으로 덮어썼을 것
#         C = {1: 79.25, 2: 101.02, 3: 92.01}
#         NOISE = 0.11 * (C[2]-C[1]) / 10.0
# ============================================================
print(f'사용 계수 C = {C}   NOISE = {NOISE:.3f}%p')
print(f'  g(활엽-침엽) = {C[2]-C[1]:.2f}   '
      f'δ(혼효-중간값) = {C[3]-(C[1]+C[2])/2:+.2f}\n')

def decompose(y, p):
    cm  = confusion_matrix(y, p, labels=[1, 2, 3])
    tot = sum((y == v).sum() * C[v] for v in (1, 2, 3))
    rows = [(f'{NAME[i]}→{NAME[j]}', int(cm[ai, bi]), C[j] - C[i],
             int(cm[ai, bi]) * (C[j] - C[i]) / tot * 100)
            for ai, i in enumerate([1, 2, 3])
            for bi, j in enumerate([1, 2, 3]) if i != j]
    return sorted(rows, key=lambda r: -abs(r[3]))

# ---- 1. 조합별 분해 ----
summary = []
for a, b in pairs():
    d = load_pred(a, b)
    y, p = d['y_true'], d['y_pred']
    rows  = decompose(y, p)
    net   = sum(r[3] for r in rows)
    gross = sum(abs(r[3]) for r in rows)
    off   = (1 - abs(net) / gross) * 100 if gross else float('nan')
    chk   = carbon_err(y, p)
    flag  = '' if abs(net - chk) < 1e-6 else f'  ⚠분해불일치 {chk:+.2f}'
    summary.append((f'{a[:3]}→{b[:3]}', net, gross, off, a == b))
    print(f'\n=== {a[:3]}→{b[:3]}  순 {net:+.2f}%  총 {gross:.2f}%  '
          f'상쇄율 {off:.0f}% ==={flag}')
    for nm, n, dc, ct in rows[:4]:
        print(f'  {nm}  n={n:6d}  Δ계수 {dc:+6.1f}  기여 {ct:+6.2f}%')

# ---- 2. 상쇄율 요약 (4-4) ----
print('\n' + '=' * 56)
print(f'{"조합":12s}{"순":>8s}{"총":>8s}{"상쇄율":>9s}')
for nm, net, gross, off, inner in sorted(summary, key=lambda r: r[3]):
    print(f'{nm:12s}{net:+8.2f}{gross:8.2f}{off:8.0f}%'
          f'{"   ← 지역 내" if inner else ""}')

IN  = [s for s in summary if s[4]]
OUT = [s for s in summary if not s[4]]
if IN and OUT:
    gi, go = np.mean([s[2] for s in IN]), np.mean([s[2] for s in OUT])
    ni, no = np.mean([abs(s[1]) for s in IN]), np.mean([abs(s[1]) for s in OUT])
    print(f'\n상쇄율 평균   지역 내 {np.mean([s[3] for s in IN]):.0f}%'
          f'   vs   지역 간 {np.mean([s[3] for s in OUT]):.0f}%')
    print(f'총오차 평균   지역 내 {gi:.2f}%   지역 간 {go:.2f}%   → {go/gi:.2f}배')
    print(f'순오차 평균   지역 내 {ni:.2f}%   지역 간 {no:.2f}%   → {no/ni:.1f}배')
    print('★ 총량은 조금 늘고 순오차는 크게 는다 = 상쇄 붕괴가 주원인')
    og = sorted(s[2] for s in OUT)
    print(f'  지역 간 총오차 분포 {[f"{v:.2f}" for v in og]}'
          f'  ← 최대값만 이탈하는지 확인')

# ---- 3. 클래스별 예측 편향 ----
print('\n' + '=' * 56)
print(f'{"조합":12s}{"침엽":>9s}{"활엽":>9s}{"혼효":>9s}   ← 예측 − 실제 (%p)')
for a, b in pairs():
    d = load_pred(a, b)
    y, p = d['y_true'], d['y_pred']
    bias = [((p == v).mean() - (y == v).mean()) * 100 for v in (1, 2, 3)]
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{v:+9.2f}' for v in bias))

# ---- 4. 2요인 분해 + 가법모델 판정 ----
M = np.full((3, 3), np.nan)      # 순오차
G = np.full((3, 3), np.nan)      # 총오차
for i, a in enumerate(REGIONS):
    for j, b in enumerate(REGIONS):
        if have(a, b):
            d = load_pred(a, b)
            M[i, j] = carbon_err(d['y_true'], d['y_pred'])
            G[i, j] = sum(abs(t[3]) for t in decompose(d['y_true'], d['y_pred']))

if not np.isnan(M).any():
    rm, cm_, gm = M.mean(1), M.mean(0), M.mean()
    print('\n' + '=' * 56)
    print(f'{"학습＼평가":12s}' + ''.join(f'{b[:3]:>9s}' for b in REGIONS)
          + f'{"행효과":>10s}')
    for i, a in enumerate(REGIONS):
        print(f'{a[:3]:12s}' + ''.join(f'{M[i,j]:+9.2f}' for j in range(3))
              + f'{rm[i]-gm:+10.2f}')
    print(f'{"열효과":12s}' + ''.join(f'{cm_[j]-gm:+9.2f}' for j in range(3))
          + f'{gm:+10.2f}  ← 전체평균')
    print('(행효과·열효과는 전체평균에서의 편차)')

    RES = M - (rm[:, None] + cm_[None, :] - gm)
    print(f'\n잔차 (실측 − 가법모델)   최대 |{np.abs(RES).max():.2f}|')
    for i, a in enumerate(REGIONS):
        print(f'{a[:3]:12s}' + ''.join(f'{RES[i,j]:+9.2f}' for j in range(3)))

    dia = np.array([RES[i, i] for i in range(3)])
    ofd = np.array([RES[i, j] for i in range(3) for j in range(3) if i != j])
    gap = ofd.mean() - dia.mean()
    print(f'\n대각 평균   {dia.mean():+.3f}'
          f'  ({"모두 음수" if (dia < 0).all() else "혼재"})')
    print(f'비대각 평균 {ofd.mean():+.3f}'
          f'  ({"모두 양수" if (ofd > 0).all() else "혼재"})')
    print(f'격차 {gap:.3f} = 노이즈({NOISE:.3f})의 {gap/NOISE:.1f}배')
    print('→ ' + ('비가법 상호작용 존재 (가법모델 기각)'
                  if (dia < 0).all() and (ofd > 0).all() and gap > 2*NOISE
                  else '상호작용 미검출'))
    print('  대각이 계통적으로 음수 = 학습·평가 동일 시 오차가 추가로 줄어듦')
    oi = np.mean([100*(1-abs(M[i,i])/G[i,i]) for i in range(3)])
    oo = np.mean([100*(1-abs(M[i,j])/G[i,j])
                  for i in range(3) for j in range(3) if i != j])
    print(f'  이 상호작용 항이 곧 4-4 의 상쇄율 차이 '
          f'(지역 내 {oi:.0f}% vs 지역 간 {oo:.0f}%)')

    c1 = M[0, 1] + M[1, 2] + M[2, 0]
    c2 = M[0, 2] + M[2, 1] + M[1, 0]
    print(f'\n순환1 {c1:+.2f}   순환2 {c2:+.2f}   차이 {abs(c1-c2):.2f}')
    print('★ 순환검정은 비대각만 쓰므로 위 구조를 검출 못 함. 단독 판정 금지')


=== dae→dae  순 -1.05%  총 4.69%  상쇄율 77% ===
  활엽→혼효  n= 15160  Δ계수  -9.0  기여  -1.01%
  활엽→침엽  n=  6016  Δ계수 -21.8  기여  -0.96%
  혼효→침엽  n=  9562  Δ계수 -12.8  기여  -0.90%
  침엽→혼효  n=  8810  Δ계수 +12.8  기여  +0.83%

=== dae→hon  순 +2.73%  총 5.58%  상쇄율 51% ===
  침엽→활엽  n= 18072  Δ계수 +21.8  기여  +2.86%
  혼효→활엽  n= 10391  Δ계수  +9.0  기여  +0.68%
  활엽→침엽  n=  3994  Δ계수 -21.8  기여  -0.63%
  침엽→혼효  n=  6545  Δ계수 +12.8  기여  +0.61%

=== dae→sun  순 -1.57%  총 5.58%  상쇄율 72% ===
  활엽→침엽  n= 10214  Δ계수 -21.8  기여  -1.67%
  침엽→혼효  n= 15968  Δ계수 +12.8  기여  +1.53%
  활엽→혼효  n= 19100  Δ계수  -9.0  기여  -1.29%
  혼효→침엽  n=  6426  Δ계수 -12.8  기여  -0.62%

=== hon→dae  순 -1.51%  총 5.64%  상쇄율 73% ===
  활엽→혼효  n= 21339  Δ계수  -9.0  기여  -1.42%
  활엽→침엽  n=  8311  Δ계수 -21.8  기여  -1.33%
  침엽→혼효  n= 14049  Δ계수 +12.8  기여  +1.32%
  혼효→침엽  n=  8772  Δ계수 -12.8  기여  -0.82%

=== hon→hon  순 +0.12%  총 4.78%  상쇄율 98% ===
  침엽→활엽  n=  8755  Δ계수 +21.8  기여  +1.37%
  활엽→침엽  n=  7102  Δ계수 -21.8  기여  -1.12%
  활엽→혼효  n= 12748  Δ계수  -9.0  기여  -0.

### `[8]` 타일(사업지) 단위 오차총량은 상쇄된 값이라 실사용 지표가 못 됩니다. `순천→홍천`은 5km 타일의 **68.7%가 오차 3% 초과**입니다.

In [ ]:
# ============================================================
# [8] 타일별 오차 — LCA는 사업지 단위로 쓰이므로 총량보다 이쪽이 실사용 지표
# ============================================================
def coarse(t, k):
    return (t // 10000 // k) * 10000 + (t % 10000 // k)

def tile_err(d, k=1, min_n=300):
    t = coarse(d['tile'], k) if k > 1 else d['tile']
    out = []
    for tt in np.unique(t):
        m = (t == tt)
        if m.sum() >= min_n:
            out.append(carbon_err(d['y_true'][m], d['y_pred'][m]))
    return np.array(out)

cache = {(a, b): load_pred(a, b) for a, b in pairs()}

# ---- 0. 타일 수 진단 ----
print('=== 유효 타일 수 (표본이 적으면 표준편차를 믿을 수 없음) ===')
print(f'{"조합":12s}{"전체":>7s}{"5km":>7s}{"10km":>7s}{"20km":>7s}{"평가px":>10s}')
for (a, b), d in cache.items():
    n_all = np.unique(d['tile']).size
    ns = [tile_err(d, k).size for k in (1, 2, 4)]
    print(f'{a[:3]}→{b[:3]:8s}{n_all:7d}' + ''.join(f'{n:7d}' for n in ns)
          + f'{d["y_true"].size:10,d}')
print('\n타일 15개 미만이면 표준편차 불확실성 ±20% 이상 → 조합 간 비교 불가')

# ---- 1. 분포 ----
scale = {}
for k, km in [(1, 5), (2, 10), (4, 20)]:
    print(f'\n--- 타일 {km}km (최소 300px) ---')
    print(f'{"조합":12s}{"타일수":>7s}{"평균":>8s}{"표준편차":>9s}'
          f'{"최소":>8s}{"최대":>8s}{"|e|>3%":>9s}')
    for (a, b), d in cache.items():
        e = tile_err(d, k)
        if e.size < 2:
            print(f'{a[:3]}→{b[:3]:8s}{e.size:7d}   (표본 부족)')
            continue
        scale[(a, b, km)] = (e.size, e.std(), (abs(e) > 3).mean() * 100)
        print(f'{a[:3]}→{b[:3]:8s}{e.size:7d}{e.mean():+8.2f}{e.std():9.2f}'
              f'{e.min():+8.2f}{e.max():+8.2f}{(abs(e)>3).mean()*100:8.1f}%')

# ---- 2. 총량 대비 타일 분산: 상쇄가 어디서 일어나는가 ----
print('\n=== 총량 오차 vs 타일 분산 (5km) ===')
print(f'{"조합":12s}{"총량":>8s}{"타일평균":>10s}{"타일SD":>9s}'
      f'{"SD/|총량|":>10s}{"타일수":>7s}')
for (a, b), d in cache.items():
    e = tile_err(d, 1)
    if e.size < 2:
        continue
    tot = carbon_err(d['y_true'], d['y_pred'])
    ratio = e.std() / abs(tot) if abs(tot) > 0.05 else float('nan')
    print(f'{a[:3]}→{b[:3]:8s}{tot:+8.2f}{e.mean():+10.2f}{e.std():9.2f}'
          f'{ratio:10.1f}{e.size:7d}')
print('SD/|총량| 이 크면 총량이 작아도 개별 사업지 위험은 큼 (상쇄에 가려진 것)')

# ---- 3. 스케일 의존성 ----
print('\n=== 스케일 의존성 (표준편차) ===')
print(f'{"조합":12s}{"5km":>10s}{"10km":>10s}{"20km":>10s}   (괄호=타일수)')
for a, b in cache:
    cells = []
    for km in (5, 10, 20):
        v = scale.get((a, b, km))
        cells.append(f'{v[1]:.2f}({v[0]})' if v else '-')
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{c:>10s}' for c in cells))
print('타일을 키울수록 SD가 줄면 국소 상쇄, 안 줄면 계통 편향 (더 나쁨)')

=== 유효 타일 수 (표본이 적으면 표준편차를 믿을 수 없음) ===
조합               전체    5km   10km   20km      평가px
dae→dae          12     10      7      3   150,000
dae→hon         105     83     31     10   150,000
dae→sun          51     47     16      4   150,000
hon→dae          37     32     11      4   150,000
hon→hon          32     31     23     10   150,000
hon→sun          51     47     16      4   150,000
sun→dae          37     32     11      4   150,000
sun→hon         105     83     31     10   150,000
sun→sun          16     15      9      4   150,000

타일 15개 미만이면 표준편차 불확실성 ±20% 이상 → 조합 간 비교 불가

--- 타일 5km (최소 300px) ---
조합              타일수      평균     표준편차      최소      최대   |e|>3%
dae→dae          10   -0.56     1.21   -2.37   +1.14     0.0%
dae→hon          83   +1.86     1.48   -1.44   +6.45    16.9%
dae→sun          47   -1.28     0.83   -3.10   +0.97     2.1%
hon→dae          32   -1.30     1.25   -4.67   +1.72     9.4%
hon→hon          31   +0.11     1.13   -2.38   +3.64     3.2%
hon→sun

In [ ]:
# ============================================================
# [9] 계수 감도 — 현재 결과는 계수차 10(55/65) 기준의 하한선
#   실제 목재기본밀도 침엽 0.35~0.51 < 활엽 0.36~0.72 로 격차가 더 큼
#   주의: 오차는 계수차에 비례하지 않고 포화함 (분모에도 들어감)
# ============================================================
def err_with(d, cc):
    t = sum((d['y_true'] == v).sum() * cc[v] for v in (1, 2, 3))
    p = sum((d['y_pred'] == v).sum() * cc[v] for v in (1, 2, 3))
    return (p - t) * 100 / t

# ---- 1. 침엽-활엽 계수차 (혼효는 중간값 고정) ----
GAPS = (0, 5, 10, 15, 20, 25, 30)
print('=== 침엽-활엽 계수차 (혼효 = 중간값) ===')
print(f'{"조합":12s}' + ''.join(f'{g:>8d}' for g in GAPS))
for a, b in pairs():
    d = load_pred(a, b)
    row = [err_with(d, {1: 55.0, 2: 55.0 + g, 3: 55.0 + g / 2}) for g in GAPS]
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{v:+8.2f}' for v in row))
print('g=0 이 전부 0.00, g=10 이 [7] 순오차와 일치해야 정상')

# ---- 2. 혼효 계수 이탈 (침엽 55 / 활엽 65 고정) ----
OFFS = (-6, -4, -2, 0, +2, +4, +6)
print('\n=== 혼효 계수 = 60 + δ (중간값에서 벗어날 때) ===')
print(f'{"조합":12s}' + ''.join(f'{o:>+8d}' for o in OFFS) + f'{"진폭":>9s}')
amp = {}
for a, b in pairs():
    d = load_pred(a, b)
    row = [err_with(d, {1: 55.0, 2: 65.0, 3: 60.0 + o}) for o in OFFS]
    amp[(a, b)] = max(row) - min(row)
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{v:+8.2f}' for v in row)
          + f'{amp[(a,b)]:9.2f}')
print(f'\n혼효 δ=±6 일 때 최대 진폭 {max(amp.values()):.2f}%p '
      f'({max(amp, key=amp.get)[0][:3]}→{max(amp, key=amp.get)[1][:3]})')
print('진폭이 작으면 혼효 계수 산정 방법이 결론에 영향 없음 → 감도분석으로 방어 가능')

# ---- 3. 부호 안정성: 계수를 어떻게 잡아도 결론이 유지되는가 ----
print('\n=== 부호 안정성 (g=5~30, 혼효 δ=-6~+6 전 조합) ===')
print(f'{"조합":12s}{"최소":>9s}{"최대":>9s}{"부호일관":>10s}')
for a, b in pairs():
    d = load_pred(a, b)
    vals = [err_with(d, {1: 55.0, 2: 55.0 + g, 3: 55.0 + g / 2 + o})
            for g in (5, 10, 15, 20, 25, 30) for o in OFFS]
    same = all(v > 0 for v in vals) or all(v < 0 for v in vals)
    print(f'{a[:3]}→{b[:3]:8s}{min(vals):+9.2f}{max(vals):+9.2f}'
          f'{"O" if same else "X":>10s}')
print('전부 O 면 "계수를 어떻게 정해도 과대/과소 방향은 바뀌지 않는다" 주장 가능')


=== 침엽-활엽 계수차 (혼효 = 중간값) ===
조합                 0       5      10      15      20      25      30
dae→dae        +0.00   -0.42   -0.81   -1.16   -1.49   -1.79   -2.08
dae→hon        +0.00   +1.01   +1.92   +2.75   +3.51   +4.21   +4.86
dae→sun        +0.00   -0.70   -1.34   -1.94   -2.51   -3.03   -3.52
hon→dae        +0.00   -0.66   -1.26   -1.81   -2.32   -2.79   -3.23
hon→hon        +0.00   -0.01   -0.02   -0.02   -0.03   -0.03   -0.04
hon→sun        +0.00   -1.32   -2.55   -3.69   -4.75   -5.74   -6.67
sun→dae        +0.00   +1.12   +2.16   +3.10   +3.98   +4.79   +5.55
sun→hon        +0.00   +2.20   +4.21   +6.03   +7.70   +9.24  +10.66
sun→sun        +0.00   +0.27   +0.51   +0.74   +0.96   +1.16   +1.34
g=0 이 전부 0.00, g=10 이 [7] 순오차와 일치해야 정상

=== 혼효 계수 = 60 + δ (중간값에서 벗어날 때) ===
조합                -6      -4      -2      +0      +2      +4      +6       진폭
dae→dae        -1.37   -1.18   -0.99   -0.81   -0.63   -0.45   -0.27     1.10
dae→hon        +2.11   +2.04   +1.98   +1.92   +

## 메커니즘 — 낙엽송### `[10]` 낙엽송 기여도학습 지역의 낙엽송 노출도(홍천 33% / 대전 8.6% / 순천 0%)와 오분류율(37.8% / 62.1% / **96.2%**)이 정확히 단조 대응합니다.> 단, 이건 **관찰**입니다. 세 지역은 위도·기후·오프셋으로도 같은 순서라 점 3개로는 구별 불가. 검증은 `[11]`/`[18]`.

---## 원인 1 — 낙엽송낙엽송은 침엽수인데 겨울에 잎을 떨굽니다.낙엽송을 본 적 없는 모델은 이걸 활엽으로 떨구고 **탄소를 과대추정**합니다.

In [ ]:
# ============================================================
# [10] 낙엽송 기여도 — 5-2 가설이 탄소 오차까지 설명하는가
#   sp 는 평가 지역 기준이므로 홍천 평가 3조합만 통과 (dae/hon/sun → hon)
#   → 노출도 대조군: hon→hon(낙엽송 학습함) vs dae/sun→hon(안 함)
# ============================================================
LARCH = 13
tbl = []
for a, b in pairs():
    d = load_pred(a, b)
    y, p, sp_ = d['y_true'], d['y_pred'], d['sp']
    con = y == 1
    mL  = con & (sp_ == LARCH)
    if mL.sum() < 500:
        continue
    tot = sum((y == v).sum() * C[v] for v in (1, 2, 3))
    print(f'\n=== {a[:3]}→{b[:3]} ===')
    contrib = {}
    for nm, m in [('낙엽송', mL), ('기타침엽', con & (sp_ != LARCH))]:
        n = m.sum()
        contrib[nm] = sum((C[v] - C[1]) * (p[m] == v).sum() for v in (2, 3)) / tot * 100
        print(f'  {nm:6s} n={n:7d}  오분류 {(p[m]!=1).mean()*100:5.1f}%  '
              f'→활엽 {(p[m]==2).mean()*100:5.1f}%  →혼효 {(p[m]==3).mean()*100:5.1f}%  '
              f'탄소기여 {contrib[nm]:+6.2f}%')
    s = sum(abs(v) for v in contrib.values())
    area = mL.sum() / con.sum() * 100
    if s:
        share = abs(contrib['낙엽송']) / s * 100
        tbl.append((f'{a[:3]}→{b[:3]}', (p[mL] != 1).mean() * 100,
                    (p[con & (sp_ != LARCH)] != 1).mean() * 100, share, area))
        print(f'  침엽 탄소오차 중 낙엽송 비중 {share:.0f}% (면적 비중 {area:.0f}%)')
        print(f'  검산: 침엽→활엽 + 침엽→혼효 기여 합 = {sum(contrib.values()):+.2f}%')

if tbl:
    print('\n' + '=' * 62)
    print(f'{"조합":12s}{"낙엽송오분류":>13s}{"기타침엽":>11s}{"배율":>7s}'
          f'{"탄소비중":>10s}{"면적비중":>10s}')
    for nm, mL_, mO_, sh, ar in tbl:
        print(f'{nm:12s}{mL_:12.1f}%{mO_:10.1f}%{mL_/mO_:7.2f}'
              f'{sh:9.0f}%{ar:9.0f}%')
    print('\n→ 탄소비중 > 면적비중 이면 낙엽송이 면적 이상으로 오차를 만드는 것')
    print('→ hon→hon 배율이 dae/sun→hon 보다 낮아야 노출도 효과 (5-2 유지)')
else:
    print('낙엽송 500px 이상 조합 없음')


=== dae→dae ===
  낙엽송    n=   2606  오분류  49.7%  →활엽  38.0%  →혼효  11.7%  탄소기여  +0.13%
  기타침엽   n=  58352  오분류  18.9%  →활엽   4.3%  →혼효  14.6%  탄소기여  +0.75%
  침엽 탄소오차 중 낙엽송 비중 14% (면적 비중 4%)
  검산: 침엽→활엽 + 침엽→혼효 기여 합 = +0.88%

=== dae→hon ===
  낙엽송    n=  21000  오분류  62.1%  →활엽  56.9%  →혼효   5.2%  탄소기여  +1.37%
  기타침엽   n=  35379  오분류  32.7%  →활엽  17.3%  →혼효  15.4%  탄소기여  +0.97%
  침엽 탄소오차 중 낙엽송 비중 59% (면적 비중 37%)
  검산: 침엽→활엽 + 침엽→혼효 기여 합 = +2.35%

=== hon→dae ===
  낙엽송    n=   5145  오분류  42.5%  →활엽  24.2%  →혼효  18.3%  탄소기여  +0.19%
  기타침엽   n=  54898  오분류  26.4%  →활엽   2.5%  →혼효  23.9%  탄소기여  +0.88%
  침엽 탄소오차 중 낙엽송 비중 18% (면적 비중 9%)
  검산: 침엽→활엽 + 침엽→혼효 기여 합 = +1.07%

=== hon→hon ===
  낙엽송    n=  17778  오분류  37.8%  →활엽  29.4%  →혼효   8.4%  탄소기여  +0.65%
  기타침엽   n=  33308  오분류  28.8%  →활엽  10.6%  →혼효  18.2%  탄소기여  +0.72%
  침엽 탄소오차 중 낙엽송 비중 48% (면적 비중 35%)
  검산: 침엽→활엽 + 침엽→혼효 기여 합 = +1.37%

=== sun→dae ===
  낙엽송    n=   5145  오분류  98.2%  →활엽  94.3%  →혼효   3.9%  탄소기여  +0.55%
  기타침엽   n=  54898  

### `[11]` 낙엽송 노출도 통제 실험 ★★**홍천 한 지역 안에서** 학습셋의 낙엽송 비율만 조작. 지역·기후·지형·라벨 시점·학습 픽셀 수·클래스 구성·평가셋 전부 고정.관찰이 아니라 **개입**이라 인과 언어를 쓸 수 있습니다.> ⚠️ **위약 대조가 없습니다.** 낙엽송이 고정된 침엽 예산 안에서 다른 침엽수를 밀어내는 구조(`n_O = n_v − n_L`)라> "낙엽송이 들어와서"인지 "침엽 구성이 바뀌어서"인지 안 갈립니다. → 보고서 8절 2번 작업.

### `[11]` 낙엽송 노출도 통제 실험지역끼리 비교하면 "지역이 다르다"는 것 자체가 교란입니다.**홍천 한 지역 안에서** 학습셋의 낙엽송 비율만 조작하고지역·기후·지형·라벨 시점·학습 픽셀 수·클래스 구성·평가셋을 전부 고정합니다.> 관찰이 아니라 **개입**이라 인과를 말할 수 있습니다.> 다만 낙엽송이 고정된 침엽 예산 안에서 다른 침엽수를 밀어내는 구조라> **위약 대조가 없습니다.**

In [ ]:
# ============================================================
# [11] 낙엽송 노출도 실험 — 관찰을 인과로
#   홍천 학습셋의 낙엽송 비율만 조작. n과 클래스 비율은 고정.
#   → 노출도와 탄소 오차의 관계가 단조적인지 직접 확인
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

LEVELS = [0.00, 0.10, 0.20, 0.33]   # 학습셋 침엽 중 낙엽송 비율
N_TR, N_TE, SEED = 200_000, 150_000, 7
LARCH = 13

hon, sun = S['hongcheon'], S['suncheon']

# --- 홍천을 타일 단위로 공간 분할 (학습 70% / 평가 30%) ---
rng = np.random.default_rng(SEED)
tiles = np.unique(hon['tile'])
te_tiles = set(rng.choice(tiles, size=int(len(tiles) * 0.3), replace=False).tolist())
is_te = np.isin(hon['tile'], list(te_tiles))
print(f'홍천 타일 {len(tiles)}개 → 학습 {len(tiles)-len(te_tiles)} / 평가 {len(te_tiles)}')

# --- 학습 풀에서 클래스 비율 고정 ---
pool = ~is_te
base = {v: (hon['y'][pool] == v).mean() for v in (1, 2, 3)}
print('학습 풀 클래스 비율:', {k: f'{v:.3f}' for k, v in base.items()})

def build_train(level, seed):
    """낙엽송 비율만 level로 맞추고 n·클래스비율은 고정"""
    r = np.random.default_rng(seed)
    idx = []
    for v in (1, 2, 3):
        n_v = int(N_TR * base[v])
        if v == 1:
            con = np.where(pool & (hon['y'] == 1))[0]
            L = con[hon['sp'][con] == LARCH]
            O = con[hon['sp'][con] != LARCH]
            n_L = int(n_v * level)
            n_O = n_v - n_L
            if n_L > len(L) or n_O > len(O):
                raise ValueError(f'level {level}: 픽셀 부족 (L {len(L)}, O {len(O)})')
            idx += [r.choice(L, n_L, replace=False), r.choice(O, n_O, replace=False)]
        else:
            c = np.where(pool & (hon['y'] == v))[0]
            idx.append(r.choice(c, n_v, replace=False))
    return np.concatenate(idx)

def make_test(d, mask, n, seed):
    c = np.where(mask)[0]
    r = np.random.default_rng(seed)
    return r.choice(c, min(n, len(c)), replace=False)

te_hon = make_test(hon, is_te, N_TE, SEED + 1)
te_sun = make_test(sun, np.ones(len(sun['y']), bool), N_TE, SEED + 2)

# --- 실행 ---
res = []
for lv in LEVELS:
    tr = build_train(lv, SEED)
    clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                 max_features='sqrt', class_weight='balanced',
                                 n_jobs=-1, random_state=SEED)
    clf.fit(hon['X'][tr], hon['y'][tr])

    row = {'level': lv}
    for tag, d, te in [('hon', hon, te_hon), ('sun', sun, te_sun)]:
        y, p = d['y'][te], clf.predict(d['X'][te])
        row[f'{tag}_f1'] = f1_score(y, p, average='macro', labels=[1, 2, 3])
        row[f'{tag}_c']  = carbon_err(y, p)
        if tag == 'hon':
            mL = (y == 1) & (d['sp'][te] == LARCH)
            row['larch_mis'] = (p[mL] != 1).mean() * 100 if mL.sum() > 100 else np.nan
    res.append(row)
    print(f'  level {lv:.2f} 완료  '
          f'hon {row["hon_c"]:+.2f}%  sun {row["sun_c"]:+.2f}%')

# --- 결과 ---
print(f'\n{"노출도":>7s}{"→홍천 F1":>10s}{"→홍천 탄소":>11s}'
      f'{"낙엽송오분류":>13s}{"→순천 F1":>10s}{"→순천 탄소":>11s}')
for r_ in res:
    print(f'{r_["level"]*100:6.0f}%{r_["hon_f1"]:10.3f}{r_["hon_c"]:+11.2f}'
          f'{r_["larch_mis"]:12.1f}%{r_["sun_f1"]:10.3f}{r_["sun_c"]:+11.2f}')

for tag, nm in [('hon', '홍천'), ('sun', '순천')]:
    v = [r_[f'{tag}_c'] for r_ in res]
    d_ = np.diff(v)
    mono = all(x < 0 for x in d_) or all(x > 0 for x in d_)
    print(f'\n→{nm} 탄소오차 {v[0]:+.2f} → {v[-1]:+.2f}  '
          f'변화폭 {abs(v[-1]-v[0]):.2f}%p  '
          f'단조 {"O" if mono else "X"}  (노이즈 {NOISE})')

홍천 타일 105개 → 학습 74 / 평가 31
학습 풀 클래스 비율: {1: '0.365', 2: '0.506', 3: '0.129'}
  level 0.00 완료  hon +1.28%  sun -3.11%
  level 0.10 완료  hon +0.60%  sun -2.96%
  level 0.20 완료  hon +0.41%  sun -2.81%
  level 0.33 완료  hon +0.24%  sun -2.58%

    노출도    →홍천 F1     →홍천 탄소       낙엽송오분류    →순천 F1     →순천 탄소
     0%     0.599      +1.28        79.3%     0.518      -3.11
    10%     0.635      +0.60        47.7%     0.525      -2.96
    20%     0.641      +0.41        39.7%     0.532      -2.81
    33%     0.645      +0.24        33.3%     0.538      -2.58

→홍천 탄소오차 +1.28 → +0.24  변화폭 1.04%p  단조 O  (노이즈 0.11)

→순천 탄소오차 -3.11 → -2.58  변화폭 0.54%p  단조 O  (노이즈 0.11)


## 기각된 가설 3건### `[12]` 수종 다양성 — **기각**침엽 수종을 1→4종으로 늘려도 **효과 없음**(0.014%p, 노이즈의 1/8).순천 침엽 NDVI가 0.686~0.741에 몰려 있어 **수종을 늘려도 분광 다양성이 안 늘어납니다.** "수종 수"는 잘못된 대리 지표였습니다.

---## 기각된 가설낙엽송이 설명하는 크기는 전체 오차의 일부입니다. 다른 설명을 찾아봅니다.

In [ ]:
# ============================================================
# [12] 학습 지역 수종 다양성 실험 — 행효과(+1.98)의 원인 규명
#   [11]이 열효과(평가지역 낙엽송)를 잡았고, 이건 행효과(학습지역 단조로움).
#   순천 학습셋의 침엽 수종 종류만 조작 → 홍천에 적용.
#   n·클래스비율 고정. 종류가 적을수록 오차가 커지면 "좁은 규칙" 가설 확정.
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

W_NDVI   = 21          # 낙엽기 NDVI 피처 인덱스 (아래 STEP 0에서 검증)
SEED     = 11
N_TR_MAX = 200_000
N_TE     = 150_000
COVER    = 0.95        # 누적 이 비율까지의 수종을 조건으로 사용
LARCH    = 13

sun, hon = S['suncheon'], S['hongcheon']

# ---------- STEP 0. 피처 인덱스 검증 ----------
# PROJECT.md 5-2: 낙엽송 0.369 / 잣나무 0.685 / 소나무 0.632
hc = hon['y'] == 1
hL = hc & (hon['sp'] == LARCH)
print(f'[검증] 홍천 낙엽송 낙엽기NDVI 평균 {hon["X"][hL, W_NDVI].mean():.3f}  (기대 0.369)')
print(f'       홍천 기타침엽              {hon["X"][hc & ~hL, W_NDVI].mean():.3f}  (기대 0.63~0.69)')
print('       크게 다르면 W_NDVI 인덱스를 셀 [1]의 피처 순서로 고쳐야 함\n')

# ---------- STEP 1. 순천 공간 분할 + 침엽 수종 조사 ----------
rng = np.random.default_rng(SEED)
tiles = np.unique(sun['tile'])
te_t  = set(rng.choice(tiles, int(len(tiles) * 0.3), replace=False).tolist())
is_te = np.isin(sun['tile'], list(te_t))
pool  = ~is_te
print(f'순천 타일 {len(tiles)}개 → 학습 {len(tiles)-len(te_t)} / 평가 {len(te_t)}')

base = {v: (sun['y'][pool] == v).mean() for v in (1, 2, 3)}
print('학습 풀 클래스 비율:', {k: f'{v:.3f}' for k, v in base.items()})

con_idx = np.where(pool & (sun['y'] == 1))[0]
sp_con  = sun['sp'][con_idx]
codes, cnts = np.unique(sp_con, return_counts=True)
order = np.argsort(-cnts)
codes, cnts = codes[order], cnts[order]
cum = np.cumsum(cnts) / cnts.sum()

print(f'\n순천 학습풀 침엽 {len(con_idx):,}px, 수종 {len(codes)}종')
print(f'{"코드":>6s}{"픽셀":>10s}{"비율":>8s}{"누적":>8s}{"낙엽기NDVI":>12s}')
for c, n, cu in zip(codes, cnts, cum):
    m = con_idx[sp_con == c]
    print(f'{c:6d}{n:10,d}{n/cnts.sum()*100:7.1f}%{cu*100:7.1f}%'
          f'{sun["X"][m, W_NDVI].mean():12.3f}')
    if cu >= COVER:
        break

# ---------- STEP 2. 조건 = 누적 상위 k종 ----------
K = int(np.argmax(cum >= COVER)) + 1
CONDS = [codes[:k] for k in range(1, K + 1)]
avail = [np.isin(sp_con, cs).sum() for cs in CONDS]

n_con = min(min(avail), int(N_TR_MAX * base[1]))
N_TR  = int(n_con / base[1])
print(f'\n조건 {len(CONDS)}개 (상위 1~{K}종). 침엽 가용 픽셀 {avail}')
print(f'→ 학습셋 크기 N_TR = {N_TR:,} (침엽 {n_con:,}) 로 전 조건 고정')
if N_TR < 60_000:
    print('⚠ N_TR 이 작음. 상위 1종 조건을 빼거나 COVER 를 낮출 것')

# ---------- STEP 3. 학습셋 구성 ----------
def build(cs, seed):
    r = np.random.default_rng(seed)
    parts = []
    for v in (1, 2, 3):
        n_v = int(N_TR * base[v])
        if v == 1:
            c = con_idx[np.isin(sp_con, cs)]
        else:
            c = np.where(pool & (sun['y'] == v))[0]
        parts.append(r.choice(c, n_v, replace=n_v > len(c)))
    return np.concatenate(parts)

def pick(d, mask, n, seed):
    c = np.where(mask)[0]
    return np.random.default_rng(seed).choice(c, min(n, len(c)), replace=False)

te_hon = pick(hon, np.ones(len(hon['y']), bool), N_TE, SEED + 1)   # [6]과 동일: 전체
te_sun = pick(sun, is_te, N_TE, SEED + 2)

# ---------- STEP 4. 실행 ----------
res = []
for cs in CONDS:
    tr = build(cs, SEED)
    tr_con = tr[sun['y'][tr] == 1]
    row = {'k': len(cs), 'codes': cs,
           'sd': sun['X'][tr_con, W_NDVI].std(),
           'tiles': np.unique(sun['tile'][tr_con]).size}

    clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                 max_features='sqrt', class_weight='balanced',
                                 n_jobs=-1, random_state=SEED)
    clf.fit(sun['X'][tr], sun['y'][tr])

    for tag, d, te in [('hon', hon, te_hon), ('sun', sun, te_sun)]:
        y, p = d['y'][te], clf.predict(d['X'][te])
        row[f'{tag}_f1'] = f1_score(y, p, average='macro', labels=[1, 2, 3])
        row[f'{tag}_c']  = carbon_err(y, p)
        if tag == 'hon':
            mL = (y == 1) & (d['sp'][te] == LARCH)
            mO = (y == 1) & (d['sp'][te] != LARCH)
            row['mis_L'] = (p[mL] != 1).mean() * 100
            row['mis_O'] = (p[mO] != 1).mean() * 100   # ← 핵심 지표
    res.append(row)
    print(f'  {len(cs)}종 완료  →홍천 {row["hon_c"]:+.2f}%  '
          f'기타침엽오분류 {row["mis_O"]:.1f}%')

# ---------- STEP 5. 결과 ----------
print(f'\n{"수종수":>6s}{"침엽SD":>9s}{"타일":>7s}{"→홍천F1":>10s}{"→홍천탄소":>11s}'
      f'{"낙엽송오분":>12s}{"기타침엽오분":>13s}{"→순천F1":>10s}{"→순천탄소":>11s}')
for r_ in res:
    print(f'{r_["k"]:6d}{r_["sd"]:9.4f}{r_["tiles"]:7d}{r_["hon_f1"]:10.3f}'
          f'{r_["hon_c"]:+11.2f}{r_["mis_L"]:11.1f}%{r_["mis_O"]:12.1f}%'
          f'{r_["sun_f1"]:10.3f}{r_["sun_c"]:+11.2f}')

def mono(v):
    d_ = np.diff(v)
    return 'O' if (all(x < 0 for x in d_) or all(x > 0 for x in d_)) else 'X'

for key, nm in [('hon_c', '→홍천 탄소'), ('mis_O', '기타침엽 오분류'),
                ('sd', '학습 침엽 SD')]:
    v = [r_[key] for r_ in res]
    print(f'\n{nm:16s} {v[0]:+.3f} → {v[-1]:+.3f}  '
          f'변화폭 {abs(v[-1]-v[0]):.3f}  단조 {mono(v)}')

print(f'\n[검산] 전 수종 조건 → 홍천 {res[-1]["hon_c"]:+.2f}%  '
      f'([6] sun→hon = +4.21, 노이즈 {NOISE})')
print(f'[교란] 침엽 학습 타일 수 {res[0]["tiles"]} → {res[-1]["tiles"]}  '
      f'(크게 줄면 수종이 아니라 공간범위 효과일 수 있음)')

[검증] 홍천 낙엽송 낙엽기NDVI 평균 0.381  (기대 0.369)
       홍천 기타침엽              0.625  (기대 0.63~0.69)
       크게 다르면 W_NDVI 인덱스를 셀 [1]의 피처 순서로 고쳐야 함

순천 타일 51개 → 학습 36 / 평가 15
학습 풀 클래스 비율: {1: '0.522', 2: '0.363', 3: '0.115'}

순천 학습풀 침엽 216,036px, 수종 7종
    코드        픽셀      비율      누적     낙엽기NDVI
    11   105,446   48.8%   48.8%       0.741
    17    47,540   22.0%   70.8%       0.686
    14    34,424   15.9%   86.7%       0.717
    15    27,817   12.9%   99.6%       0.698

조건 4개 (상위 1~4종). 침엽 가용 픽셀 [np.int64(105446), np.int64(152986), np.int64(187410), np.int64(215227)]
→ 학습셋 크기 N_TR = 199,999 (침엽 104,435) 로 전 조건 고정
  1종 완료  →홍천 +4.26%  기타침엽오분류 49.3%
  2종 완료  →홍천 +4.34%  기타침엽오분류 52.7%
  3종 완료  →홍천 +4.30%  기타침엽오분류 51.0%
  4종 완료  →홍천 +4.28%  기타침엽오분류 50.0%

   수종수     침엽SD     타일     →홍천F1      →홍천탄소       낙엽송오분       기타침엽오분     →순천F1      →순천탄소
     1   0.0786     35     0.483      +4.26       96.8%        49.3%     0.606      +1.55
     2   0.1041     35     0.478      +4.34       97.1%        52

### `[13]` 분광 커버리지 — 부분 확인, 일반화 실패커버율과 오차의 상관은 −0.58 수준. 다음 셀에서 **미커버량으로 부호를 예측하려는 시도는 실패**합니다.못 덮은 픽셀이 *어느 클래스로* 떨어지는지를 **양**이 결정하지 않기 때문입니다.

In [ ]:
# ============================================================
# [13] 분광 커버리지 — 학습 분포가 평가 분포를 얼마나 덮는가
#   [12]에서 수종 "수"는 원인이 아님이 밝혀짐. 범위가 진짜 변수인지 확인.
#   학습 불필요. S 의 원 분포와 [6] 탄소 오차를 대조.
# ============================================================
W_NDVI = 21
LARCH  = 13
Q = (2.5, 97.5)

def cover(a, b, cls=1):
    """a 학습셋 클래스 분포가 b 평가셋의 같은 클래스를 덮는 비율"""
    xa = S[a]['X'][S[a]['y'] == cls, W_NDVI]
    xb = S[b]['X'][S[b]['y'] == cls, W_NDVI]
    lo, hi = np.percentile(xa, Q)
    return ((xb >= lo) & (xb <= hi)).mean() * 100, lo, hi

print(f'{"조합":12s}{"침엽커버":>10s}{"활엽커버":>10s}{"혼효커버":>10s}'
      f'{"탄소오차":>10s}{"학습범위(침엽)":>20s}')
rows = []
for a in REGIONS:
    for b in REGIONS:
        if not have(a, b): continue
        cv = [cover(a, b, c)[0] for c in (1, 2, 3)]
        _, lo, hi = cover(a, b, 1)
        d = load_pred(a, b)
        ce = carbon_err(d['y_true'], d['y_pred'])
        rows.append((f'{a[:3]}→{b[:3]}', cv[0], cv[1], cv[2], ce, a == b))
        print(f'{a[:3]}→{b[:3]:8s}{cv[0]:9.1f}%{cv[1]:9.1f}%{cv[2]:9.1f}%'
              f'{ce:+10.2f}   [{lo:.3f}, {hi:.3f}]')

# 커버율과 탄소 오차의 관계
cv1 = np.array([r[1] for r in rows])
ce  = np.array([r[4] for r in rows])
print(f'\n침엽 커버율 vs 탄소오차   상관 {np.corrcoef(cv1, ce)[0,1]:+.3f}')
print(f'침엽 커버율 vs |탄소오차|  상관 {np.corrcoef(cv1, np.abs(ce))[0,1]:+.3f}')

# 홍천 평가 3조합만: 낙엽송이 덮이는가
print(f'\n--- 홍천 평가 3조합, 낙엽송 커버율 ---')
hL = (S['hongcheon']['y'] == 1) & (S['hongcheon']['sp'] == LARCH)
xL = S['hongcheon']['X'][hL, W_NDVI]
for a in REGIONS:
    xa = S[a]['X'][S[a]['y'] == 1, W_NDVI]
    lo, hi = np.percentile(xa, Q)
    c = ((xL >= lo) & (xL <= hi)).mean() * 100
    d = load_pred(a, 'hongcheon')
    y, p, sp_ = d['y_true'], d['y_pred'], d['sp']
    m = (y == 1) & (sp_ == LARCH)
    print(f'{a[:3]}→hon   낙엽송커버 {c:5.1f}%   '
          f'낙엽송오분류 {(p[m]!=1).mean()*100:5.1f}%   '
          f'탄소 {carbon_err(y, p):+.2f}%')

조합                침엽커버      활엽커버      혼효커버      탄소오차            학습범위(침엽)
dae→dae          95.0%     95.0%     95.0%     -0.81   [0.365, 0.779]
dae→hon          69.0%     75.2%     85.0%     +1.92   [0.365, 0.779]
dae→sun          68.2%     86.1%     79.9%     -1.34   [0.365, 0.779]
hon→dae          99.7%     95.9%     96.5%     -1.26   [0.238, 0.824]
hon→hon          95.0%     95.0%     95.0%     -0.02   [0.238, 0.824]
hon→sun          91.6%     83.5%     78.2%     -2.55   [0.238, 0.824]
sun→dae          88.0%     84.5%     86.3%     +2.16   [0.456, 0.855]
sun→hon          59.8%     45.2%     70.0%     +4.21   [0.456, 0.855]
sun→sun          95.0%     95.0%     95.0%     +0.51   [0.456, 0.855]

침엽 커버율 vs 탄소오차   상관 -0.580
침엽 커버율 vs |탄소오차|  상관 -0.663

--- 홍천 평가 3조합, 낙엽송 커버율 ---
dae→hon   낙엽송커버  48.3%   낙엽송오분류  62.1%   탄소 +1.92%
hon→hon   낙엽송커버  95.3%   낙엽송오분류  37.8%   탄소 -0.02%
sun→hon   낙엽송커버  18.4%   낙엽송오분류  96.2%   탄소 +4.21%


In [ ]:
# 미커버 픽셀의 순 불균형 (면적 가중)
for a in REGIONS:
    for b in REGIONS:
        if not have(a, b): continue
        un = {}
        for c in (1, 2):
            xa = S[a]['X'][S[a]['y'] == c, W_NDVI]
            xb = S[b]['X'][S[b]['y'] == c, W_NDVI]
            lo, hi = np.percentile(xa, Q)
            un[c] = (~((xb >= lo) & (xb <= hi))).mean() * (S[b]['y'] == c).mean() * 100
        d = load_pred(a, b)
        print(f'{a[:3]}→{b[:3]:8s} 미커버침엽 {un[1]:5.2f}%  미커버활엽 {un[2]:5.2f}%  '
              f'차 {un[1]-un[2]:+6.2f}  탄소 {carbon_err(d["y_true"], d["y_pred"]):+.2f}%')

dae→dae      미커버침엽  1.99%  미커버활엽  1.99%  차  +0.00  탄소 -0.81%
dae→hon      미커버침엽 11.65%  미커버활엽 12.19%  차  -0.53  탄소 +1.92%
dae→sun      미커버침엽 16.50%  미커버활엽  5.28%  차 +11.22  탄소 -1.34%
hon→dae      미커버침엽  0.11%  미커버활엽  1.62%  차  -1.51  탄소 -1.26%
hon→hon      미커버침엽  1.88%  미커버활엽  2.45%  차  -0.57  탄소 -0.02%
hon→sun      미커버침엽  4.37%  미커버활엽  6.24%  차  -1.88  탄소 -2.55%
sun→dae      미커버침엽  4.79%  미커버활엽  6.16%  차  -1.37  탄소 +2.16%
sun→hon      미커버침엽 15.15%  미커버활엽 26.88%  차 -11.73  탄소 +4.21%
sun→sun      미커버침엽  2.60%  미커버활엽  1.90%  차  +0.70  탄소 +0.51%


## 주장 ③ — 지역 오프셋### `[14]` 상록활엽수 탐색낙엽송의 거울 사례("상록성 활엽")를 찾으려 했으나 순천 폴리곤 70개(0.3%)로 사실상 없음.**대신 여기서 지역 오프셋이 드러났습니다.**

---## 원인 2 — 지역 오프셋 *(핵심)***같은 입력이 지역마다 다른 정답을 가집니다.**이건 라벨 오류가 아닙니다. 같은 수종도 위도·기후 때문에 실제로 다르게 찍힙니다.

In [ ]:
# ============================================================
# [14] 상록활엽수 탐색 — 낙엽송의 거울 사례
#   5-2가 "침엽인데 겨울 NDVI 낮은 수종"(낙엽송)을 찾았다면,
#   이건 "활엽인데 겨울 NDVI 높은 수종"(상록활엽수)을 찾는다.
#   순천 선정 당시의 원 가설이었으나 미검증 상태.
# ============================================================
W_NDVI = 21
MIN_PX = 5000

for rg in REGIONS:
    d = S[rg]
    print(f'\n{"="*62}\n=== {rg} ===')
    for cls, nm in [(2, '활엽'), (1, '침엽')]:
        m = d['y'] == cls
        ref = np.median(d['X'][m, W_NDVI])
        codes, cnts = np.unique(d['sp'][m], return_counts=True)
        rows = []
        for c, n in zip(codes, cnts):
            if n < MIN_PX:
                continue
            v = np.median(d['X'][m & (d['sp'] == c), W_NDVI])
            rows.append((c, n, n / m.sum() * 100, v, v - ref))
        rows.sort(key=lambda r: -r[3])
        print(f'\n  [{nm}] 클래스 중앙값 {ref:.3f}   ({m.sum():,}px)')
        print(f'  {"코드":>6s}{"픽셀":>10s}{"비율":>8s}{"겨울NDVI":>10s}{"편차":>9s}')
        for c, n, p, v, dv in rows:
            flag = ''
            if cls == 2 and dv > 0.15:  flag = '  ← 상록활엽수 의심'
            if cls == 1 and dv < -0.15: flag = '  ← 낙엽성 침엽 의심'
            print(f'  {c:6d}{n:10,d}{p:7.1f}%{v:10.3f}{dv:+9.3f}{flag}')

# 5-2 검산: 홍천 낙엽송(13)이 침엽 목록에서 편차 -0.2 이하로 잡혀야 정상


=== daejeon ===

  [활엽] 클래스 중앙값 0.440   (238,972px)
      코드        픽셀      비율    겨울NDVI       편차
      49     5,429    2.3%     0.457   +0.017
      30    68,756   28.8%     0.446   +0.006
      34    87,366   36.6%     0.438   -0.002
      32    20,644    8.6%     0.436   -0.004
      31    29,814   12.5%     0.435   -0.005
      33    22,374    9.4%     0.433   -0.007

  [침엽] 클래스 중앙값 0.655   (239,240px)
      코드        픽셀      비율    겨울NDVI       편차
      14   140,138   58.6%     0.672   +0.017
      12     7,112    3.0%     0.657   +0.002
      11    62,711   26.2%     0.657   +0.002
      17     5,486    2.3%     0.494   -0.161  ← 낙엽성 침엽 의심
      13    20,557    8.6%     0.409   -0.246  ← 낙엽성 침엽 의심

=== hongcheon ===

  [활엽] 클래스 중앙값 0.369   (294,060px)
      코드        픽셀      비율    겨울NDVI       편차
      33    28,885    9.8%     0.415   +0.046
      34    62,197   21.2%     0.385   +0.016
      37    17,994    6.1%     0.367   -0.002
      32    63,827   21.7%     0.358   -0.011
  

### `[15]` 지역 오프셋 ★★★ — 보고서 4-③낙엽기 NDVI 클래스 중앙값:| 지역 | 활엽 | 침엽 | 경계 ||---|---|---|---|| 홍천 | 0.369 | 0.538 | 0.453 || 순천 | 0.515 | 0.742 | 0.629 |**순천의 활엽(0.515)이 홍천의 침엽(0.538)과 거의 같습니다.** 지역 간 차이(0.204) > 클래스 간 차이(0.169).이 경계선 계산만으로 오차 방향을 **실측과 상관 0.933, 전이 6조합 부호 전부 일치**로 맞춥니다.> 🔑 **이 결과는 RandomForest와 무관합니다.** 라벨 데이터의 기술통계 + 경계선 논증이므로 어떤 분류기에도 성립합니다.

### `[15]` 지역 오프셋 — 클래스 중앙값 이동만으로 오차가 예측되는가**지역 간 차이가 클래스 간 차이보다 큽니다.**한 지역에서 배운 경계선을 다른 지역에 가져가면 클래스가 통째로 밀립니다.> 이 결과는 **분류기와 무관합니다.** 라벨 데이터의 기술통계 + 경계선 논증이므로> 어떤 모델을 써도 성립합니다.

In [ ]:
# ============================================================
# [15] 지역 오프셋 가설 — 클래스 중앙값 이동만으로 오차가 예측되는가
#   [14] 발견: 순천 활엽(0.515) ≈ 홍천 침엽(0.538)
#   학습 지역의 결정경계를 평가 지역 분포에 대보면 오분류 방향이 나온다.
# ============================================================
W_NDVI = 21
OUT = '/content/drive/MyDrive/forest/outputs'
C = {1: 55.0, 2: 65.0, 3: 60.0}   # 이 셀 출력은 구 계수 기준(보고서와 일치)

D = {}
for rg in REGIONS:
    z = np.load(f'{OUT}/{rg}_samples.npz')
    D[rg] = {'y': z['y'], 'w': z['X'][:, W_NDVI]}

med = {rg: {c: np.median(D[rg]['w'][D[rg]['y'] == c]) for c in (1, 2, 3)}
       for rg in REGIONS}
print(f'{"지역":12s}{"활엽":>9s}{"침엽":>9s}{"경계":>9s}')
for rg in REGIONS:
    b = (med[rg][2] + med[rg][1]) / 2
    print(f'{rg:12s}{med[rg][2]:9.3f}{med[rg][1]:9.3f}{b:9.3f}')

print(f'\n{"조합":12s}{"경계":>8s}{"침엽오분":>10s}{"활엽오분":>10s}'
      f'{"예측오차":>10s}{"실측오차":>10s}')
pred, act = [], []
for a in REGIONS:
    b = (med[a][2] + med[a][1]) / 2
    for e in REGIONS:
        if not have(a, e): continue
        wy, ww = D[e]['y'], D[e]['w']
        # 학습 경계 기준: 침엽인데 경계 아래 → 활엽행 / 활엽인데 위 → 침엽행
        f1 = (ww[wy == 1] < b).mean()
        f2 = (ww[wy == 2] > b).mean()
        p1, p2 = (wy == 1).mean(), (wy == 2).mean()
        tot = sum((wy == v).mean() * C[v] for v in (1, 2, 3))
        pe = (f1 * p1 * 10 - f2 * p2 * 10) / tot * 100
        d = load_pred(a, e)
        ae = carbon_err(d['y_true'], d['y_pred'])
        pred.append(pe); act.append(ae)
        print(f'{a[:3]}→{e[:3]:8s}{b:8.3f}{f1*100:9.1f}%{f2*100:9.1f}%'
              f'{pe:+10.2f}{ae:+10.2f}')

pred, act = np.array(pred), np.array(act)
print(f'\n상관 {np.corrcoef(pred, act)[0,1]:+.3f}   '
      f'부호일치 {(np.sign(pred)==np.sign(act)).sum()}/{len(pred)}')

C = dict(C_CONF)   # ★ 전역 계수 복구 — 이후 셀이 구 계수로 돌지 않도록


지역                 활엽       침엽       경계
daejeon         0.440    0.655    0.548
hongcheon       0.369    0.538    0.453
suncheon        0.515    0.742    0.629

조합                경계      침엽오분      활엽오분      예측오차      실측오차
dae→dae        0.548     22.5%     15.0%     +0.50     -0.81
dae→hon        0.548     51.4%      8.8%     +2.48     +1.92
dae→sun        0.548      7.9%     38.0%     -1.74     -1.34
hon→dae        0.453     11.7%     43.6%     -2.12     -1.26
hon→hon        0.453     39.1%     22.8%     +0.58     -0.02
hon→sun        0.453      2.4%     77.6%     -4.75     -2.55
sun→dae        0.629     40.5%      4.5%     +2.39     +2.16
sun→hon        0.629     63.5%      3.0%     +3.71     +4.21
sun→sun        0.629     17.5%     18.5%     +0.34     +0.51

상관 +0.933   부호일치 7/9


### `[16]` 지역별 정규화 — **실패**오프셋을 빼서 교정하려 했으나 **전이 6조합 전부 부호 반전**, 1개는 악화. 교정이 아니라 **과보정**입니다.원인 ① 상관된 피처를 개별 정규화해 RF가 쓰던 피처 간 관계가 깨짐원인 ② 중앙값이 클래스 구성에 오염 — 순천은 침엽이 47.6%라 중앙값이 침엽 쪽으로 끌립니다.이걸 빼면 **"순천에 침엽이 많다"는 진짜 정보까지** 지워집니다.→ 이게 **조건부 이동**입니다. 라벨 없이 피처만 정렬해서는 원리적으로 교정 불가.

---## 처방 시도 — 지역별 정규화각 지역의 **위성 통계만으로**(라벨 없이) 기준선을 맞춰봅니다.라벨이 필요 없으므로 갱신이 밀린 지역에도 쓸 수 있다는 것이 이 방법의 장점입니다.

In [ ]:
# ============================================================
# [16] 지역별 정규화 — 오프셋 가설의 처방 검증
#   [14][15]에서 지역 간 클래스 중앙값 차이(0.204)가
#   클래스 간 차이(0.169)보다 크다는 것이 확인됨.
#   각 지역 통계로 피처를 정규화하면 축이 정렬되어 전이 오차가 줄어야 한다.
#   라벨 불필요 — 영상 통계만 사용하므로 실무 적용 가능.
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

SEED   = 16
N_TR   = 200_000
N_TE   = 150_000
W_NDVI = 21
C      = {1: 55.0, 2: 65.0, 3: 60.0}   # 이 셀 출력은 구 계수 기준(보고서와 일치)
NOISE  = 0.11

def carbon_err(y_true, y_pred):
    t = sum((y_true == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((y_pred == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100 / t

# ---------- 지역별 통계 (라벨 없이 전체 픽셀에서) ----------
STAT = {}
for rg in REGIONS:
    X = S[rg]['X']
    STAT[rg] = (np.median(X, axis=0), X.std(axis=0) + 1e-8)

print(f'{"지역":12s}{"활엽":>9s}{"침엽":>9s}{"경계":>9s}{"전체중앙":>10s}')
for rg in REGIONS:
    y, w = S[rg]['y'], S[rg]['X'][:, W_NDVI]
    m2, m1 = np.median(w[y == 2]), np.median(w[y == 1])
    print(f'{rg:12s}{m2:9.3f}{m1:9.3f}{(m1+m2)/2:9.3f}{STAT[rg][0][W_NDVI]:10.3f}')

def norm(rg, idx):
    med, sd = STAT[rg]
    return (S[rg]['X'][idx] - med) / sd

# ---------- 정규화 후 중앙값이 정렬되는지 먼저 확인 ----------
print(f'\n[정렬 확인] 정규화 후 클래스 중앙값 (낙엽기 NDVI 축)')
print(f'{"지역":12s}{"활엽":>9s}{"침엽":>9s}{"경계":>9s}')
for rg in REGIONS:
    y = S[rg]['y']
    z = (S[rg]['X'][:, W_NDVI] - STAT[rg][0][W_NDVI]) / STAT[rg][1][W_NDVI]
    m2, m1 = np.median(z[y == 2]), np.median(z[y == 1])
    print(f'{rg:12s}{m2:9.3f}{m1:9.3f}{(m1+m2)/2:9.3f}')
print('→ 세 지역 경계가 서로 가까워지면 정규화가 축을 정렬한 것')

# ---------- 학습/평가 인덱스 (raw·norm 공통, 동일 시드) ----------
IDX = {}
for rg in REGIONS:
    r = np.random.default_rng(SEED)
    tiles = np.unique(S[rg]['tile'])
    te_t  = set(r.choice(tiles, int(len(tiles) * 0.3), replace=False).tolist())
    is_te = np.isin(S[rg]['tile'], list(te_t))
    tr = r.choice(np.where(~is_te)[0], min(N_TR, (~is_te).sum()), replace=False)
    IDX[rg] = {
        'tr': tr,
        'te_self': r.choice(np.where(is_te)[0], min(N_TE, is_te.sum()), replace=False),
        'te_all':  r.choice(len(S[rg]['y']), N_TE, replace=False),
    }

# ---------- 실행 ----------
def run(mode):
    out = {}
    for a in REGIONS:
        Xtr = norm(a, IDX[a]['tr']) if mode == 'norm' else S[a]['X'][IDX[a]['tr']]
        clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                     max_features='sqrt', class_weight='balanced',
                                     n_jobs=-1, random_state=SEED)
        clf.fit(Xtr, S[a]['y'][IDX[a]['tr']])
        for b in REGIONS:
            te = IDX[b]['te_self'] if a == b else IDX[b]['te_all']
            Xte = norm(b, te) if mode == 'norm' else S[b]['X'][te]
            y, p = S[b]['y'][te], clf.predict(Xte)
            out[(a, b)] = (f1_score(y, p, average='macro', labels=[1, 2, 3]),
                           carbon_err(y, p))
        print(f'  {a[:3]} 학습 완료 ({mode})')
    return out

print(f'\n--- raw 실행 ---');  raw = run('raw')
print(f'--- norm 실행 ---');   nrm = run('norm')

# ---------- 결과 ----------
print(f'\n{"조합":12s}{"F1_raw":>9s}{"F1_nrm":>9s}{"ΔF1":>8s}'
      f'{"C_raw":>9s}{"C_nrm":>9s}{"|C|감소":>10s}')
gain_in, gain_out = [], []
for a in REGIONS:
    for b in REGIONS:
        (f0, c0), (f1_, c1) = raw[(a, b)], nrm[(a, b)]
        red = abs(c0) - abs(c1)
        (gain_in if a == b else gain_out).append((f1_ - f0, red))
        mark = '  ← 지역 내' if a == b else ''
        print(f'{a[:3]}→{b[:3]:8s}{f0:9.3f}{f1_:9.3f}{f1_-f0:+8.3f}'
              f'{c0:+9.2f}{c1:+9.2f}{red:+10.2f}{mark}')

for nm, g in [('지역 내', gain_in), ('지역 간', gain_out)]:
    df = np.mean([x[0] for x in g]); dc = np.mean([x[1] for x in g])
    print(f'\n{nm}  ΔF1 평균 {df:+.3f}   |탄소오차| 평균 감소 {dc:+.2f}%p')

k = ('suncheon', 'hongcheon')
print(f'\n[핵심] sun→hon  탄소 {raw[k][1]:+.2f} → {nrm[k][1]:+.2f}   '
      f'F1 {raw[k][0]:.3f} → {nrm[k][0]:.3f}')
print(f'       감소폭이 노이즈 {NOISE} 를 크게 넘으면 처방이 작동한 것')

C = dict(C_CONF)   # ★ 전역 계수 복구 — 이후 셀이 구 계수로 돌지 않도록


지역                 활엽       침엽       경계      전체중앙
daejeon         0.440    0.655    0.548     0.533
hongcheon       0.369    0.538    0.453     0.418
suncheon        0.515    0.742    0.629     0.657

[정렬 확인] 정규화 후 클래스 중앙값 (낙엽기 NDVI 축)
지역                 활엽       침엽       경계
daejeon        -0.717    0.945    0.114
hongcheon      -0.294    0.725    0.215
suncheon       -1.067    0.637   -0.215
→ 세 지역 경계가 서로 가까워지면 정규화가 축을 정렬한 것

--- raw 실행 ---
  dae 학습 완료 (raw)
  hon 학습 완료 (raw)
  sun 학습 완료 (raw)
--- norm 실행 ---
  dae 학습 완료 (norm)
  hon 학습 완료 (norm)
  sun 학습 완료 (norm)

조합             F1_raw   F1_nrm     ΔF1    C_raw    C_nrm     |C|감소
dae→dae         0.629    0.629  -0.000    -0.93    -0.93     +0.00  ← 지역 내
dae→hon         0.566    0.533  -0.034    +1.59    -1.73     -0.15
dae→sun         0.540    0.589  +0.049    -1.79    +1.20     +0.59
hon→dae         0.582    0.565  -0.016    -1.11    +1.94     -0.83
hon→hon         0.633    0.633  -0.000    +0.57    +0.57     -0.00  ← 지역 내
hon→sun 

In [ ]:
# ============================================================
# [17] 축 선정 검증 — 낙엽기 NDVI가 정말 지배적 축인가
#   지역 간 차이를 클래스 간 차이로 나눈 비율이 클수록 전이에 해로운 축
# ============================================================
rows = []
for i in range(38):
    gap_r = max(np.median(S[rg]['X'][:, i]) for rg in REGIONS) - \
            min(np.median(S[rg]['X'][:, i]) for rg in REGIONS)
    gap_c, sd = [], []
    for rg in REGIONS:
        x, y = S[rg]['X'][:, i], S[rg]['y']
        gap_c.append(abs(np.median(x[y == 1]) - np.median(x[y == 2])))
        sd.append(x.std())
    rows.append((i, gap_r, np.mean(gap_c), gap_r / (np.mean(gap_c) + 1e-8),
                 np.mean(gap_c) / (np.mean(sd) + 1e-8)))

rows.sort(key=lambda r: -r[3])
print(f'{"피처":>5s}{"지역간차":>10s}{"클래스간차":>12s}'
      f'{"비율(해로움)":>14s}{"판별력":>10s}')
for i, gr, gc, ratio, disc in rows[:12]:
    mark = '  ← 낙엽기 NDVI' if i == 21 else ''
    print(f'{i:5d}{gr:10.3f}{gc:12.3f}{ratio:14.2f}{disc:10.3f}{mark}')

print(f'\n낙엽기 NDVI(21) 순위: '
      f'{[r[0] for r in rows].index(21) + 1}/38')
print('비율 > 1 이면 지역 차이가 클래스 차이보다 큰 축 (전이에 해로움)')
print('판별력이 높으면서 비율이 낮은 축이 이상적')

   피처      지역간차       클래스간차       비율(해로움)       판별력
    1     0.006       0.001          7.81     0.061
    3     0.011       0.004          2.50     0.258
   37     0.017       0.008          2.30     0.224
   29     0.015       0.007          2.12     0.126
   35     0.013       0.006          1.99     0.198
   31     0.013       0.007          1.92     0.215
   10     0.019       0.011          1.73     0.563
    0     0.002       0.001          1.71     0.139
   11     0.014       0.009          1.65     0.470
   32     0.196       0.120          1.63     1.099
   33     0.016       0.011          1.52     0.276
   36     0.211       0.141          1.50     1.212

낙엽기 NDVI(21) 순위: 14/38
비율 > 1 이면 지역 차이가 클래스 차이보다 큰 축 (전이에 해로움)
판별력이 높으면서 비율이 낮은 축이 이상적


### `[18]` 노출도 곡선 정밀화 (4점 → 8점)**"N%면 충분"이 성립하지 않습니다.** 개선 50%에 노출도 8%, 90%에 33%. 임계값이 없습니다.> 안전한 주장은 **끝점 간 1.04%p 감소**(노이즈의 9배)까지입니다.> 구간별 단조성은 8→10% 구간이 −0.05%p로 **노이즈 이하**라 주장하지 않습니다.> `[11]`과의 "검산 일치"는 **같은 시드·같은 분할**이므로 코드 정확성 확인이지 안정성 확인이 아닙니다.

### `[18]` 노출도 곡선 정밀화 (4점 → 8점)안전한 주장은 **끝점 간 감소폭**까지입니다.구간별 단조성은 일부 구간이 노이즈 이하라 주장하지 않습니다.> `[11]` 과의 검산 일치는 **같은 시드·같은 분할**이므로> 코드 정확성 확인이지 안정성 확인이 아닙니다.

In [ ]:
# ============================================================
# [18] 노출도 곡선 정밀화 — [11]의 4점을 8점으로
#   목적: "10%면 충분"의 근거 확보 또는 기각
#   [11]과 동일 시드·동일 분할이라 4개 점은 재현되어야 함(검산)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

LEVELS = [0.00, 0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.33]
N_TR, N_TE, SEED = 200_000, 150_000, 7      # [11]과 동일
LARCH = 13

hon, sun = S['hongcheon'], S['suncheon']

rng = np.random.default_rng(SEED)
tiles = np.unique(hon['tile'])
te_tiles = set(rng.choice(tiles, size=int(len(tiles) * 0.3), replace=False).tolist())
is_te = np.isin(hon['tile'], list(te_tiles))
pool = ~is_te
base = {v: (hon['y'][pool] == v).mean() for v in (1, 2, 3)}
print(f'홍천 타일 {len(tiles)} → 학습 {len(tiles)-len(te_tiles)} / 평가 {len(te_tiles)}')
print('학습 풀 클래스 비율:', {k: f'{v:.3f}' for k, v in base.items()})

con_all = np.where(pool & (hon['y'] == 1))[0]
L_all = con_all[hon['sp'][con_all] == LARCH]
O_all = con_all[hon['sp'][con_all] != LARCH]
n_con = int(N_TR * base[1])
print(f'침엽 학습 {n_con:,}px 중 낙엽송 최대 {min(len(L_all), n_con)/n_con*100:.1f}% 까지 가능')

def build_train(level, seed):
    r = np.random.default_rng(seed)
    idx = []
    for v in (1, 2, 3):
        n_v = int(N_TR * base[v])
        if v == 1:
            n_L = int(round(n_v * level))
            n_O = n_v - n_L
            if n_L > len(L_all) or n_O > len(O_all):
                raise ValueError(f'level {level}: 픽셀 부족')
            idx += [r.choice(L_all, n_L, replace=False),
                    r.choice(O_all, n_O, replace=False)]
        else:
            c = np.where(pool & (hon['y'] == v))[0]
            idx.append(r.choice(c, n_v, replace=False))
    return np.concatenate(idx)

def pick(d, mask, n, seed):
    c = np.where(mask)[0]
    return np.random.default_rng(seed).choice(c, min(n, len(c)), replace=False)

te_hon = pick(hon, is_te, N_TE, SEED + 1)
te_sun = pick(sun, np.ones(len(sun['y']), bool), N_TE, SEED + 2)

res = []
for lv in LEVELS:
    tr = build_train(lv, SEED)
    clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                 max_features='sqrt', class_weight='balanced',
                                 n_jobs=-1, random_state=SEED)
    clf.fit(hon['X'][tr], hon['y'][tr])
    row = {'lv': lv}
    for tag, d, te in [('hon', hon, te_hon), ('sun', sun, te_sun)]:
        y, p = d['y'][te], clf.predict(d['X'][te])
        row[f'{tag}_f1'] = f1_score(y, p, average='macro', labels=[1, 2, 3])
        row[f'{tag}_c']  = carbon_err(y, p)
        if tag == 'hon':
            mL = (y == 1) & (d['sp'][te] == LARCH)
            row['mis'] = (p[mL] != 1).mean() * 100
    res.append(row)
    print(f'  {lv*100:5.1f}%  홍천 {row["hon_c"]:+.2f}%  '
          f'순천 {row["sun_c"]:+.2f}%  낙엽송오분 {row["mis"]:.1f}%')

# ---------- 결과 ----------
print(f'\n{"노출도":>7s}{"홍천F1":>9s}{"홍천탄소":>10s}{"낙엽송오분":>12s}'
      f'{"순천F1":>9s}{"순천탄소":>10s}{"홍천개선률":>12s}')
c0, cN = res[0]['hon_c'], res[-1]['hon_c']
for r_ in res:
    frac = (c0 - r_['hon_c']) / (c0 - cN) * 100 if c0 != cN else np.nan
    print(f'{r_["lv"]*100:6.1f}%{r_["hon_f1"]:9.3f}{r_["hon_c"]:+10.2f}'
          f'{r_["mis"]:11.1f}%{r_["sun_f1"]:9.3f}{r_["sun_c"]:+10.2f}'
          f'{frac:11.0f}%')

# ---------- [11] 검산 ----------
print('\n[검산] [11] 4점과 일치해야 함 (동일 시드·분할)')
ref = {0.00: (+1.28, 79.3), 0.10: (+0.60, 47.7),
       0.20: (+0.41, 39.7), 0.33: (+0.24, 33.3)}
for r_ in res:
    if r_['lv'] in ref:
        rc, rm = ref[r_['lv']]
        d1, d2 = r_['hon_c'] - rc, r_['mis'] - rm
        ok = 'O' if abs(d1) < NOISE else 'X'
        print(f'  {r_["lv"]*100:5.1f}%  탄소 {r_["hon_c"]:+.2f} vs {rc:+.2f} '
              f'(Δ{d1:+.2f}) {ok}   오분류 {r_["mis"]:.1f} vs {rm:.1f} (Δ{d2:+.1f})')

# ---------- 포화점 판정 ----------
print('\n[포화점] 개선률이 임계에 처음 도달하는 노출도')
for th in (50, 65, 80, 90):
    hit = next((r_['lv']*100 for r_ in res
                if (c0 - r_['hon_c']) / (c0 - cN) * 100 >= th), None)
    print(f'  {th}% 개선 → {hit if hit is not None else "미도달"}% 노출')

d_ = np.diff([r_['hon_c'] for r_ in res])
print(f'\n단조 {"O" if all(x < 0 for x in d_) else "X"}   '
      f'구간별 변화 ' + ' '.join(f'{x:+.2f}' for x in d_))
print(f'가장 큰 감소 구간: '
      f'{LEVELS[int(np.argmin(d_))]*100:.0f}→{LEVELS[int(np.argmin(d_))+1]*100:.0f}%')

홍천 타일 105 → 학습 74 / 평가 31
학습 풀 클래스 비율: {1: '0.365', 2: '0.506', 3: '0.129'}
침엽 학습 72,924px 중 낙엽송 최대 88.8% 까지 가능
    0.0%  홍천 +1.28%  순천 -3.11%  낙엽송오분 79.3%
    2.0%  홍천 +1.04%  순천 -3.05%  낙엽송오분 67.9%
    5.0%  홍천 +0.80%  순천 -2.98%  낙엽송오분 56.5%
    8.0%  홍천 +0.65%  순천 -3.01%  낙엽송오분 50.0%
   10.0%  홍천 +0.60%  순천 -2.96%  낙엽송오분 47.7%
   15.0%  홍천 +0.46%  순천 -2.90%  낙엽송오분 42.7%
   20.0%  홍천 +0.39%  순천 -2.87%  낙엽송오분 39.4%
   33.0%  홍천 +0.22%  순천 -2.63%  낙엽송오분 33.2%

    노출도     홍천F1      홍천탄소       낙엽송오분     순천F1      순천탄소       홍천개선률
   0.0%    0.599     +1.28       79.3%    0.518     -3.11          0%
   2.0%    0.613     +1.04       67.9%    0.523     -3.05         23%
   5.0%    0.625     +0.80       56.5%    0.524     -2.98         45%
   8.0%    0.632     +0.65       50.0%    0.524     -3.01         59%
  10.0%    0.635     +0.60       47.7%    0.525     -2.96         64%
  15.0%    0.638     +0.46       42.7%    0.528     -2.90         77%
  20.0%    0.641     +0.39       39.4%    0.5

---## 계수 확정과 검산

In [ ]:
# ============================================================
# [19] 계수 감도의 닫힌 형태 — [9] 격자 스캔을 대체
#
#   e(%) = Σ b_v·C_v / Σ f_v·C_v
#     b : 예측편향 (%p, 예측비율 − 실제비율).  Σb = 0
#     f : 평가지역의 실제 클래스 비율
#
#   Σb=0 이므로 C 전체에 상수를 더해도 분자는 불변 → 분모만 커짐(포화의 정체)
#   계수 확정 후에는 plug_in(C) 한 줄. [7]~[10] 재실행 불필요.
#
#   [9] 철회 사항: 음수 GAPS 스캔은 하지 않는다.
#     g<0 을 범위에 넣으면 g=0 을 가로질러 전 조합이 자동 X.
#     결과가 미리 정해진 검사라 정보 없음. 부호는 g>0 안에서만 물어야 함.
# ============================================================
import numpy as np

def bias_frac(d):
    """편향벡터 b(%p)와 평가지역 실제 클래스비율 f."""
    n = len(d['y_true'])
    f = np.array([(d['y_true'] == v).sum() / n for v in (1, 2, 3)])
    p = np.array([(d['y_pred'] == v).sum() / n for v in (1, 2, 3)])
    return (p - f) * 100.0, f

def e_of(b, f, C):
    """닫힌 형태. C = [침엽, 활엽, 혼효]. 절대 스케일 아닌 상대차만 의미 있음."""
    C = np.asarray(C, dtype=float)
    return float(b @ C / (f @ C))

def C_gd(g, delta, base=55.0):
    """침엽=base, 활엽=base+g, 혼효=중간값+delta."""
    return [base, base + g, base + g / 2.0 + delta]

BF = {ab: bias_frac(load_pred(*ab)) for ab in pairs()}
KEYS = list(BF)

# ---- 0. 검산: 닫힌 형태가 [9]/[7] 과 일치하는가 ----
print('=== 검산 (g=10, δ=0 → 4-3 표) ===')
print(f'{"조합":12s}{"닫힌형태":>10s}{"brute":>9s}{"차":>8s}')
for a, b_ in KEYS:
    bb, ff = BF[(a, b_)]
    v1 = e_of(bb, ff, C_gd(10, 0))
    v2 = err_with(load_pred(a, b_), {1: 55.0, 2: 65.0, 3: 60.0})
    print(f'{a[:3]}→{b_[:3]:8s}{v1:+10.2f}{v2:+9.2f}{v1-v2:+8.4f}')
print('차가 전부 0.0000 이어야 정상')

# ---- 1. 계수차 g 를 실제 있을 법한 구간에서 ----
GG = (1, 2, 3, 5, 8, 10, 15, 20)
print('\n=== 침엽-활엽 계수차 g (혼효 = 중간값) ===')
print(f'{"조합":12s}' + ''.join(f'{g:>7d}' for g in GG))
for a, b_ in KEYS:
    bb, ff = BF[(a, b_)]
    print(f'{a[:3]}→{b_[:3]:8s}'
          + ''.join(f'{e_of(bb, ff, C_gd(g, 0)):+7.2f}' for g in GG))
print(f'노이즈 기준선 {NOISE}%p — 이 값을 넘는 칸만 실재')

# ---- 2. 노이즈를 넘는 최소 g (조합별 검출 하한) ----
print('\n=== |e| > 0.11%p 가 되는 최소 g ===')
grid = np.arange(0.1, 30.01, 0.1)
for a, b_ in KEYS:
    bb, ff = BF[(a, b_)]
    hit = [g for g in grid if abs(e_of(bb, ff, C_gd(g, 0))) > NOISE]
    print(f'{a[:3]}→{b_[:3]:8s}' +
          (f'g ≥ {hit[0]:.1f}' if hit else 'g≤30 에서 검출 불가'))

# ---- 3. 혼효 계수 이탈 δ 의 기여 (4-6 공식의 누락항) ----
print('\n=== δ 항 vs g 항 (g=10 고정) ===')
print(f'{"조합":12s}{"b3":>8s}{"δ=-6":>8s}{"δ=0":>8s}{"δ=+6":>8s}'
      f'{"진폭":>8s}{"근사":>8s}')
for a, b_ in KEYS:
    bb, ff = BF[(a, b_)]
    row = [e_of(bb, ff, C_gd(10, dd)) for dd in (-6, 0, 6)]
    amp_ = max(row) - min(row)
    print(f'{a[:3]}→{b_[:3]:8s}{bb[2]:+8.2f}'
          + ''.join(f'{v:+8.2f}' for v in row)
          + f'{amp_:8.2f}{12*abs(bb[2])/(ff@np.array(C_gd(10,0))):8.2f}')
print('진폭 ≈ 12·|b3| / 평균계수. 근사인 이유는 δ가 분모에도 들어가기 때문')
print('→ 4-6 "혼효 기여 0" 은 δ=0 에서만 참. 칼날 위 결과임')

# ---- 4. 부호 안정성 — g>0 안에서만 물음 ----
print('\n=== 부호 안정성 (g ∈ [2,20], δ ∈ [-6,+6]) ===')
print(f'{"조합":12s}{"최소":>9s}{"최대":>9s}{"부호":>6s}{"|min|>노이즈":>12s}')
for a, b_ in KEYS:
    bb, ff = BF[(a, b_)]
    vals = [e_of(bb, ff, C_gd(g, dd))
            for g in (2, 3, 5, 8, 10, 15, 20) for dd in (-6, -3, 0, 3, 6)]
    same = all(v > 0 for v in vals) or all(v < 0 for v in vals)
    safe = same and min(abs(v) for v in vals) > NOISE
    print(f'{a[:3]}→{b_[:3]:8s}{min(vals):+9.2f}{max(vals):+9.2f}'
          f'{"O" if same else "X":>6s}{"O" if safe else "X":>12s}')
print('부호 O 만으로는 부족. 최솟값이 노이즈를 넘어야 주장 가능')
print('g<0 이면 부호 전부 반전, 크기는 근사 대칭 (분모 효과로 약간 큼)')

# ---- 5. 계수 확정 후 — 이 함수만 호출 ----
def plug_in(C_dict, label=''):
    """확정 계수 대입. C_dict = {1: ..., 2: ..., 3: ...}"""
    C = [C_dict[1], C_dict[2], C_dict[3]]
    print(f'\n=== plug_in {label} C={C} ===')
    for a, b_ in KEYS:
        bb, ff = BF[(a, b_)]
        v = e_of(bb, ff, C)
        flag = '' if abs(v) > NOISE else '  (노이즈 이하)'
        print(f'{a[:3]}→{b_[:3]:8s}{v:+8.2f}%{flag}')

plug_in({1: 55.0, 2: 65.0, 3: 60.0}, '임시값(현행)')
# plug_in({1: 103.0, 2: 105.0, 3: 104.0}, '강원대 거친계산')   # ← 계수 확보 후 주석 해제

=== 검산 (g=10, δ=0 → 4-3 표) ===
조합                닫힌형태    brute       차
dae→dae          -0.81    -0.81 -0.0000
dae→hon          +1.92    +1.92 -0.0000
dae→sun          -1.34    -1.34 +0.0000
hon→dae          -1.26    -1.26 -0.0000
hon→hon          -0.02    -0.02 +0.0000
hon→sun          -2.55    -2.55 +0.0000
sun→dae          +2.16    +2.16 -0.0000
sun→hon          +4.21    +4.21 -0.0000
sun→sun          +0.51    +0.51 -0.0000
차가 전부 0.0000 이어야 정상

=== 침엽-활엽 계수차 g (혼효 = 중간값) ===
조합                1      2      3      5      8     10     15     20
dae→dae       -0.09  -0.17  -0.26  -0.42  -0.66  -0.81  -1.16  -1.49
dae→hon       +0.21  +0.41  +0.62  +1.01  +1.56  +1.92  +2.75  +3.51
dae→sun       -0.14  -0.29  -0.42  -0.70  -1.09  -1.34  -1.94  -2.51
hon→dae       -0.14  -0.27  -0.40  -0.66  -1.02  -1.26  -1.81  -2.32
hon→hon       -0.00  -0.00  -0.01  -0.01  -0.01  -0.02  -0.02  -0.03
hon→sun       -0.27  -0.54  -0.80  -1.32  -2.07  -2.55  -3.69  -4.75
sun→dae       +0.23  +0.46  +0.69 

### `[20]` 확정 계수 적용임시값 55/65/60 → **강원 NFI 2013 / GIR 79.25 / 101.02 / 92.01.**계수차가 10 → 21.77로 커져 **결론이 뒤집히지 않고 강해졌습니다.** `순천→홍천` +4.21% → **+5.92%**.

### `[20]` 확정 계수 적용임시값 55/65/60 → **강원 NFI 2013 / GIR 79.25 / 101.02 / 92.01**.계수차가 커져 결론이 뒤집히지 않고 강해집니다.

In [ ]:
# ============================================================
# [20] 계수 확정 적용 — 국가 검증계수 기반   (rev.2)
#
#   출처: Lee et al. (2015) JCCR 6(4):303-310, Table 1 (GIR 검증 공표계수)
#         강원도 NFI 2013 임상별 ha당 축적, 동 논문 Table 2
#   전환계수 = Σ(D·BEF·(1+R)·V)/ΣV → 침엽 0.7956 / 활엽 1.4113
#   혼효 = 두 값의 평균 (국가 인벤토리 50:50 규약)
#   C(tC/ha) = 축적 × 전환계수 × CF(0.5)
#
#   rev.2 수정: (1) 딕셔너리 키를 3글자→풀네임 (KeyError)
#               (2) MIN_PX 5000→500 (예측파일은 타일당 ~1400px. 5000이면 전멸)
#   ★ 한계: 강원도 값. 홍천에만 정당하고 대전·순천은 대리값
# ============================================================
import numpy as np

C_GW  = {1: 79.25, 2: 101.02, 3: 92.01}    # 강원 NFI 2013 (g=21.77, δ=+1.87)
C_OLD = {1: 55.0,  2:  65.0,  3: 60.0}     # 구 임시값 (g=10, δ=0) — 검산용
C_NAT = {1: 68.70, 2: 106.35, 3: 71.73}    # 전국 2015 (g=37.64, δ=-15.80) ★미검증

REGC = {r: C_GW for r in REGIONS}   # 시군구 축적 확보 시 지역별로 교체
G_USE, G_BASE = C_GW[2] - C_GW[1], 10.0
NOISE_G = NOISE * G_USE / G_BASE    # 노이즈도 g에 비례 [유도, 미확정]

def cerr(y_true, y_pred, C):
    t = sum((y_true == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((y_pred == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100.0 / t

D = {ab: load_pred(*ab) for ab in pairs()}
print(f'노이즈 기준선: {NOISE} × {G_USE:.2f}/{G_BASE} = {NOISE_G:.3f}%p  [유도, 재측정 필요]\n')

# ---- 1. 9조합 재계산 ----
# 근사 f 로 낸 사전 예측. 0.1%p 이내면 근사 검증됨
APPROX = {('daejeon','daejeon'):-1.04, ('daejeon','hongcheon'):+2.71,
          ('daejeon','suncheon'):-1.56, ('hongcheon','daejeon'):-1.48,
          ('hongcheon','hongcheon'):+0.12,('hongcheon','suncheon'):-3.47,
          ('suncheon','daejeon'):+3.08,  ('suncheon','hongcheon'):+5.89,
          ('suncheon','suncheon'):+0.83}

print('=== 탄소 오차: 계수 교체 ===')
print(f'{"조합":11s}{"임시(g=10)":>12s}{"강원(g=22)":>12s}{"전국(g=38)":>12s}'
      f'{"사전예측":>10s}{"차":>7s}')
NEW = {}
for a, b in pairs():
    d = D[(a, b)]
    v_old = cerr(d['y_true'], d['y_pred'], C_OLD)
    v_gw  = cerr(d['y_true'], d['y_pred'], REGC[b])
    v_nat = cerr(d['y_true'], d['y_pred'], C_NAT)
    NEW[(a, b)] = v_gw
    ap = APPROX.get((a, b), float('nan'))
    print(f'{a[:3]}→{b[:3]:7s}{v_old:+12.2f}{v_gw:+12.2f}{v_nat:+12.2f}'
          f'{ap:+10.2f}{v_gw-ap:+7.2f}')
print('임시열이 4-3 표와 일치해야 정상. 차가 0.1 이내면 근사 f 검증됨')

# ---- 2. 오차/노이즈 비율 (v4 §4-13 "검출 하한" 표 대체) ----
print('\n=== |오차| / 노이즈 ===')
print('구 §4-13 은 노이즈를 0.11 로 고정한 채 g 를 움직여 §4-1 과 모순이었음.')
print('노이즈가 g 에 비례하면 비율은 g 와 무관하게 일정 → 임계 g 개념 자체가 성립 안 함')
for ab in sorted(NEW, key=lambda k: -abs(NEW[k])):
    r = abs(NEW[ab]) / NOISE_G
    print(f'{ab[0][:3]}→{ab[1][:3]:8s}{NEW[ab]:+8.2f}%{r:9.1f}배'
          f'{"" if r > 1 else "   ← 노이즈 이하"}')

# ---- 3. 4-8 순환 제약 재검정 ----
c1 = NEW[('daejeon','hongcheon')] + NEW[('hongcheon','suncheon')] \
   + NEW[('suncheon','daejeon')]
c2 = NEW[('daejeon','suncheon')] + NEW[('suncheon','hongcheon')] \
   + NEW[('hongcheon','daejeon')]
print(f'\n=== 4-8 순환 제약 ===\n순환1 {c1:+.2f}  순환2 {c2:+.2f}  '
      f'차이 {abs(c1-c2):.2f}  기준 {NOISE_G:.3f}')
print(f'판정: {"통과" if abs(c1-c2) < NOISE_G else "재검토"}   (구 0.11 로 판정 금지)')

# ---- 4. 행효과 / 열효과 ----
print('\n=== 행효과(학습) / 열효과(평가) — 비대각만 ===')
off = [(a, b) for a, b in pairs() if a != b]
for r in REGIONS:
    row = np.mean([NEW[k] for k in off if k[0] == r])
    col = np.mean([NEW[k] for k in off if k[1] == r])
    print(f'{r:12s} 행 {row:+6.2f}   열 {col:+6.2f}')

# ---- 5. 사업지 단위 재집계 (4-7) ----
MIN_PX = 500          # 예측파일 15만px / 타일 ~105개 ≈ 1400px/타일
def tile_rates(d, C, thr=3.0, group=1):
    t = d['tile']
    key = ((t // 10000) // group) * 10000 + ((t % 10000) // group)
    out = [cerr(d['y_true'][key == k], d['y_pred'][key == k], C)
           for k in np.unique(key) if (key == k).sum() >= MIN_PX]
    out = np.array(out)
    if len(out) == 0:
        return 0, float('nan'), float('nan')
    return len(out), (np.abs(out) > thr).mean() * 100, out.std()

print(f'\n=== 4-7 사업지 |e| > 3% 비율 (MIN_PX={MIN_PX}) ===')
print(f'{"조합":11s}{"n(5km)":>8s}{"5km":>8s}{"10km":>8s}{"20km":>8s}{"SD":>8s}')
for a, b in pairs():
    d, C = D[(a, b)], REGC[b]
    n1, r1, s1 = tile_rates(d, C, group=1)
    _,  r2, _  = tile_rates(d, C, group=2)
    _,  r4, _  = tile_rates(d, C, group=4)
    print(f'{a[:3]}→{b[:3]:7s}{n1:8d}{r1:8.1f}{r2:8.1f}{r4:8.1f}{s1:8.2f}')
print('★ n 을 4-7 과 대조: 홍천평가 83 / 순천평가 47 / hon→hon 31 / dae→dae 10')
print('   어긋나면 MIN_PX 를 조정할 것. 타일 10개 미만은 정성 관찰로만')

# ---- 6. δ 감도 ----
print('\n=== δ 감도 (g=21.77 고정) ===')
print(f'{"조합":11s}' + ''.join(f'{o:>+9}' for o in (-6,-3,0,1.87,3,6)) + f'{"진폭":>8s}')
for a, b in pairs():
    base = REGC[b]; mid = (base[1] + base[2]) / 2
    row = [cerr(D[(a,b)]['y_true'], D[(a,b)]['y_pred'],
                {1: base[1], 2: base[2], 3: mid + o})
           for o in (-6, -3, 0, 1.87, 3, 6)]
    print(f'{a[:3]}→{b[:3]:7s}' + ''.join(f'{v:+9.2f}' for v in row)
          + f'{max(row)-min(row):8.2f}')
print('강원 축적 기준 δ=+1.87. 부호가 δ 에 따라 뒤집히는 조합을 확인할 것')

노이즈 기준선: 0.11 × 21.77/10.0 = 0.239%p  [유도, 재측정 필요]

=== 탄소 오차: 계수 교체 ===
조합             임시(g=10)    강원(g=22)    전국(g=38)      사전예측      차
dae→dae           -0.81       -1.05       -3.15     -1.04  -0.01
dae→hon           +1.92       +2.73       +5.29     +2.71  +0.02
dae→sun           -1.34       -1.57       -7.03     -1.56  -0.01
hon→dae           -1.26       -1.51       -6.08     -1.48  -0.03
hon→hon           -0.02       +0.12       -1.26     +0.12  -0.00
hon→sun           -2.55       -3.48       -8.81     -3.47  -0.01
sun→dae           +2.16       +3.12       +5.70     +3.08  +0.04
sun→hon           +4.21       +5.92      +12.15     +5.89  +0.03
sun→sun           +0.51       +0.84       +0.52     +0.83  +0.01
임시열이 4-3 표와 일치해야 정상. 차가 0.1 이내면 근사 f 검증됨

=== |오차| / 노이즈 ===
구 §4-13 은 노이즈를 0.11 로 고정한 채 g 를 움직여 §4-1 과 모순이었음.
노이즈가 g 에 비례하면 비율은 g 와 무관하게 일정 → 임계 g 개념 자체가 성립 안 함
sun→hon        +5.92%     24.7배
hon→sun        -3.48%     14.5배
sun→dae        +3.12%     13.0배
dae→hon        +2.7

### `[21]` 주장 검증 — **미실행**문서의 모든 수치 진술을 실행 가능한 검사로 바꾼 셀. 타일 오차 분포 그림이 여기서 나옵니다.

### `[21]` 주장 검증 — **미실행**

In [ ]:
# ============================================================
# [21] 주장 검증 — 핸드오프의 모든 수치 진술을 실행 가능한 검사로
#
#   원칙: 문서의 표에 들어가는 숫자는 이 셀의 출력에서만 나온다.
#         대화나 손계산으로 만든 숫자는 문서에 넣지 않는다.
#   계수/노이즈를 바꿀 때마다 이 셀을 돌려 어떤 주장이 깨지는지 본다.
# ============================================================
import numpy as np

C_USE = {1: 79.25, 2: 101.02, 3: 92.01}   # §6 확정값
G_USE = C_USE[2] - C_USE[1]
NOISE_BASE, G_BASE = 0.11, 10.0            # [5]/[6] 측정 당시 계수

CLAIMS = []
def claim(tag, cond, detail):
    CLAIMS.append((tag, bool(cond), detail))
    print(f'[{"OK " if cond else "FAIL"}] {tag}: {detail}')

def cerr(d, C):
    t = sum((d['y_true'] == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((d['y_pred'] == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100.0 / t

D = {ab: load_pred(*ab) for ab in pairs()}
E = {ab: cerr(D[ab], C_USE) for ab in D}

# --- 0. 노이즈 기준선 정합성 (§4-1 vs §4-13 모순의 원인) ---
noise = NOISE_BASE * G_USE / G_BASE
print(f'\n노이즈 기준선: {NOISE_BASE} × {G_USE:.2f}/{G_BASE} = {noise:.3f}%p  [유도]')
print('★ 이 비례가 맞는지는 [5]/[6] 재측정으로 확인해야 함. 미확인 상태\n')

# --- 1. 오차/노이즈 비율 (§4-13 교체용) ---
print('=== 조합별 |오차| / 노이즈 ===')
for ab in sorted(E, key=lambda k: -abs(E[k])):
    r = abs(E[ab]) / noise
    print(f'{ab[0][:3]}→{ab[1][:3]:8s}{E[ab]:+8.2f}%{r:9.1f}배'
          f'{"" if r > 1 else "   ← 노이즈 이하"}')

above = [ab for ab in E if abs(E[ab]) / noise > 1]
claim('실재성', len(above) == 8 and ('hongcheon','hongcheon') not in above,
      f'{len(above)}/9 조합이 노이즈 초과. 제외: '
      f'{[f"{a[:3]}→{b[:3]}" for a,b in E if abs(E[(a,b)])/noise<=1]}')

# --- 2. 부호 안정성 — 범위를 반드시 명시 ---
GR, DR = (2, 5, 10, 15, 21.77, 30), (-6, -3, 0, 1.87, 3, 6)
print(f'\n=== 부호 안정성  g∈{GR}  δ∈{DR} ===')
stable = []
for ab in E:
    vals = [cerr(D[ab], {1: 55.0, 2: 55.0+g, 3: 55.0+g/2+dd})
            for g in GR for dd in DR]
    ok = all(v > 0 for v in vals) or all(v < 0 for v in vals)
    if ok: stable.append(ab)
    print(f'{ab[0][:3]}→{ab[1][:3]:8s}[{min(vals):+7.2f},{max(vals):+7.2f}]'
          f'  {"O" if ok else "X"}')
claim('부호 강건', len(stable) == 3,
      f'{len(stable)}/9 — {[f"{a[:3]}→{b[:3]}" for a,b in stable]}. '
      f'★ 범위를 바꾸면 결과가 바뀜. 문서에 범위 병기 필수')

# --- 3. 실재성 ≠ 방향 (v3~v4가 혼동한 지점) ---
claim('실재성 ≠ 방향', len(above) > len(stable),
      f'존재 {len(above)}조합 vs 방향 {len(stable)}조합. '
      '"구조적 실재성"으로 합쳐 쓰지 말 것')

# --- 4. 지역 내 vs 지역 간 (4-4 의 계수 축 버전) ---
din  = [abs(E[(a,b)]) for a,b in E if a == b]
dout = [abs(E[(a,b)]) for a,b in E if a != b]
claim('지역내 < 지역간', max(din) < min(dout),
      f'지역내 최대 {max(din):.2f} vs 지역간 최소 {min(dout):.2f}')

# --- 5. δ 민감도: 부호를 뒤집는 조합 특정 ---
print('\n=== δ 가 부호를 뒤집는 조합 (g 고정) ===')
flip = []
for ab in E:
    v = [cerr(D[ab], {1: C_USE[1], 2: C_USE[2],
                      3: (C_USE[1]+C_USE[2])/2 + dd}) for dd in (-6, 6)]
    if v[0]*v[1] < 0:
        flip.append(ab); print(f'{ab[0][:3]}→{ab[1][:3]:8s}{v[0]:+7.2f} → {v[1]:+7.2f}')
claim('δ 취약 조합', set(flip) <= {('daejeon','suncheon'), ('hongcheon','daejeon')},
      f'{[f"{a[:3]}→{b[:3]}" for a,b in flip]} — 시군구 축적으로 δ 확정 필요')

# --- 요약 ---
bad = [t for t, ok, _ in CLAIMS if not ok]
print(f'\n{"="*50}\n{len(CLAIMS)-len(bad)}/{len(CLAIMS)} 통과')
if bad: print(f'★ 깨진 주장: {bad} → 핸드오프 해당 절 수정 필요')